In [2]:
import sqlite3
from pathlib import Path
import pandas as pd
import numpy as np
PROJECT_ROOT = Path.cwd().parents[1] if Path.cwd().name == "ml" else Path.cwd()
DB_PATH = PROJECT_ROOT/"backend"/"ml"/"feature_preparation.db"
print("Database exists:", DB_PATH.exists())
print("Database path:",DB_PATH)

conn = sqlite3.connect(DB_PATH)
print("Database connection successful1!")

Database exists: True
Database path: C:\Users\GARIMA YADUVANSHI\OneDrive\Desktop\DSAForge\DSAForge\backend\ml\feature_preparation.db
Database connection successful1!


In [3]:
tables = pd.read_sql_query(
         """ SELECT name 
         FROM sqlite_master
         WHERE type = 'table'
         ORDER BY name
         """,
         conn)
print("Tables available in our database:")
display(tables)

Tables available in our database:


,name
0,accepted_problem_times
1,codeforces_api_problems
2,problem_catalog
3,user_problem_features


In [4]:
#Inspect the structure of our main training-data table

features_schema = pd.read_sql_query(
    "PRAGMA table_info(user_problem_features)",
    conn
)

display(features_schema[["name","type"]])

,name,type
0,user,TEXT
1,problem,TEXT
2,attempts,INTEGER
3,accepted,INTEGER
4,wrong_attempts,INTEGER
5,first_timestamp,INTEGER
6,last_timestamp,INTEGER
7,problem_rating,REAL


In [5]:
sample_features = pd.read_sql_query(
    """
    SELECT *
    FROM user_problem_features
    LIMIT 5
    """,
    conn
)

display(sample_features)

,user,problem,attempts,accepted,wrong_attempts,first_timestamp,last_timestamp,problem_rating
0,user12423,2034G1,2,1,1,1732984962,1732987693,None
1,user12423,2034H,2,1,0,1732982312,1732982759,None
2,user12423,2034F2,1,1,0,1732981003,1732981003,None
3,user12423,2034F1,1,1,0,1732980608,1732980608,None
4,user12423,2034E,1,1,0,1732979455,1732979455,None


In [6]:
metadata_schema = pd.read_sql_query(
    "PRAGMA table_info(codeforces_api_problems)",
    conn
)

display(metadata_schema[["name", "type"]])

,name,type
0,problem_id,TEXT
1,contest_id,INTEGER
2,problem_index,TEXT
3,problem_name,TEXT
4,problem_rating,REAL
5,tags,TEXT


In [7]:
metadata_summary = pd.read_sql_query(
    """
    SELECT
        COUNT(*) AS total_problems,
        SUM(CASE WHEN problem_rating IS NOT NULL THEN 1 ELSE 0 END)
            AS problems_with_rating,
        SUM(CASE WHEN tags IS NOT NULL AND TRIM(tags) != '' THEN 1 ELSE 0 END)
            AS problems_with_tags
    FROM codeforces_api_problems
    """,
    conn
)

display(metadata_summary)

,total_problems,problems_with_rating,problems_with_tags
0,11450,11152,11450


In [8]:
print("Connected database:")
display(pd.read_sql_query("PRAGMA database_list", conn))

print("Actual number of metadata rows:")
display(pd.read_sql_query(
    "SELECT COUNT(*) AS total_rows FROM codeforces_api_problems",
    conn
))

Connected database:


,seq,name,file
0,0,main,C:\Users\GARIMA YADUVANSHI\OneDrive\Desktop\DS...


Actual number of metadata rows:


,total_rows
0,11450


In [9]:
metadata_summary = pd.read_sql_query(
    """
    SELECT
        COUNT(*) AS total_problems,
        COUNT(problem_rating) AS problems_with_rating,
        COUNT(tags) AS problems_with_tags
    FROM codeforces_api_problems
    """,
    conn
)

display(metadata_summary)

,total_problems,problems_with_rating,problems_with_tags
0,11450,11152,11450


In [10]:
import importlib.util

print(
    "LightGBM installed:",
    importlib.util.find_spec("lightgbm") is not None
)

print(
    "Scikit-learn installed:",
    importlib.util.find_spec("sklearn") is not None
)

LightGBM installed: False
Scikit-learn installed: True


In [11]:
import sys
!{sys.executable} -m pip install lightgbm

'C:\Users\GARIMA' is not recognized as an internal or external command,
operable program or batch file.


In [13]:
import lightgbm as lgb

print("LightGBM version:", lgb.__version__)

LightGBM version: 4.7.0


In [12]:
import subprocess
import sys

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "lightgbm"
])

0

In [14]:
time_summary = pd.read_sql_query(
    """
    SELECT
        COUNT(*) AS total_user_problem_pairs,
        MIN(first_timestamp) AS earliest_timestamp,
        MAX(last_timestamp) AS latest_timestamp,
        COUNT(first_timestamp) AS pairs_with_first_timestamp,
        COUNT(last_timestamp) AS pairs_with_last_timestamp
    FROM user_problem_features
    """,
    conn
)

display(time_summary)

,total_user_problem_pairs,earliest_timestamp,latest_timestamp,pairs_with_first_timestamp,pairs_with_last_timestamp
0,7780135,1266581015,1733063062,7780135,7780135


In [15]:
accepted_schema = pd.read_sql_query(
    "PRAGMA table_info(accepted_problem_times)",
    conn
)

print("Columns in accepted_problem_times:")
display(accepted_schema[["name", "type"]])

print("Sample records:")
display(pd.read_sql_query(
    "SELECT * FROM accepted_problem_times LIMIT 5",
    conn
))

Columns in accepted_problem_times:


,name,type
0,user,TEXT
1,problem,TEXT
2,accepted_at,INTEGER


Sample records:


,user,problem,accepted_at
0,user12423,2034G1,1732987693
1,user12423,2034H,1732982759
2,user12423,2034F2,1732981003
3,user12423,2034F1,1732980608
4,user12423,2034E,1732979455


In [16]:
yearly_solutions = pd.read_sql_query(
    """
    SELECT
        strftime('%Y', accepted_at, 'unixepoch') AS year,
        COUNT(*) AS accepted_records,
        COUNT(DISTINCT user) AS active_users
    FROM accepted_problem_times
    GROUP BY year
    ORDER BY year
    """,
    conn
)

display(yearly_solutions)

,year,accepted_records,active_users
0,2010,2116,47
1,2011,5965,75
2,2012,10925,104
3,2013,13784,134
4,2014,16718,173
5,2015,26929,249
6,2016,38073,364
7,2017,54657,557
8,2018,105458,918
9,2019,203160,1483


In [17]:
cutoff_check = pd.read_sql_query(
    """
    SELECT
        SUM(
            CASE WHEN accepted_at < strftime('%s', '2024-07-01')
                 THEN 1 ELSE 0 END
        ) AS earlier_solutions,

        SUM(
            CASE WHEN accepted_at >= strftime('%s', '2024-07-01')
                 THEN 1 ELSE 0 END
        ) AS later_solutions,

        COUNT(
            DISTINCT CASE
                WHEN accepted_at < strftime('%s', '2024-07-01')
                THEN user
            END
        ) AS users_before_cutoff,

        COUNT(
            DISTINCT CASE
                WHEN accepted_at >= strftime('%s', '2024-07-01')
                THEN user
            END
        ) AS users_with_later_solutions

    FROM accepted_problem_times
    """,
    conn
)

display(cutoff_check)

,earlier_solutions,later_solutions,users_before_cutoff,users_with_later_solutions
0,5638347,1627461,13564,14952


In [18]:
training_period_check = pd.read_sql_query(
    """
    SELECT
        COUNT(*) AS accepted_in_training_window,
        COUNT(DISTINCT user) AS users_with_training_window_solutions
    FROM accepted_problem_times
    WHERE accepted_at >= strftime('%s', '2023-07-01')
      AND accepted_at <  strftime('%s', '2024-01-01')
    """,
    conn
)

display(training_period_check)

,accepted_in_training_window,users_with_training_window_solutions
0,1075086,10429


In [19]:
monthly_check = pd.read_sql_query(
    """
    SELECT
        strftime('%Y-%m', accepted_at, 'unixepoch') AS month,
        COUNT(*) AS accepted_records,
        COUNT(DISTINCT user) AS active_users
    FROM accepted_problem_times
    WHERE accepted_at >= strftime('%s', '2023-01-01')
      AND accepted_at <  strftime('%s', '2024-01-01')
    GROUP BY month
    ORDER BY month
    """,
    conn
)

display(monthly_check)

,month,accepted_records,active_users
0,2023-01,113444,4932
1,2023-02,115057,4974
2,2023-03,130994,5343
3,2023-04,117311,5329
4,2023-05,129571,5480
5,2023-06,133371,5697
6,2023-07,170236,6383
7,2023-08,174850,6675
8,2023-09,179521,7213
9,2023-10,177512,7433


In [20]:
training_users_check = pd.read_sql_query(
    """
    SELECT
        COUNT(DISTINCT user) AS unique_users,
        COUNT(*) AS accepted_records
    FROM accepted_problem_times
    WHERE accepted_at >= 1688169600
      AND accepted_at < 1704067200
    """,
    conn
)

display(training_users_check)

,unique_users,accepted_records
0,10429,1075086


In [21]:
training_eligibility = pd.read_sql_query(
    """
    WITH prior_users AS (
        SELECT DISTINCT user
        FROM accepted_problem_times
        WHERE accepted_at < 1688169600
    ),
    prior_solved AS (
        SELECT user, problem
        FROM accepted_problem_times
        WHERE accepted_at < 1688169600
    ),
    future_solved AS (
        SELECT a.user, a.problem
        FROM accepted_problem_times AS a
        JOIN prior_users AS u
          ON a.user = u.user
        WHERE a.accepted_at >= 1688169600
          AND a.accepted_at < 1704067200
          AND NOT EXISTS (
              SELECT 1
              FROM prior_solved AS p
              WHERE p.user = a.user
                AND p.problem = a.problem
          )
    )
    SELECT
        COUNT(*) AS new_problems_solved,
        COUNT(DISTINCT user) AS users_with_new_solutions
    FROM future_solved
    """,
    conn
)

display(training_eligibility)

,new_problems_solved,users_with_new_solutions
0,943655,7865


In [22]:
positive_metadata_check = pd.read_sql_query(
    """
    WITH prior_users AS (
        SELECT DISTINCT user
        FROM accepted_problem_times
        WHERE accepted_at < 1688169600
    ),
    prior_solved AS (
        SELECT user, problem
        FROM accepted_problem_times
        WHERE accepted_at < 1688169600
    ),
    future_new_solutions AS (
        SELECT DISTINCT a.user, a.problem
        FROM accepted_problem_times AS a
        JOIN prior_users AS u
          ON a.user = u.user
        WHERE a.accepted_at >= 1688169600
          AND a.accepted_at < 1704067200
          AND NOT EXISTS (
              SELECT 1
              FROM prior_solved AS p
              WHERE p.user = a.user
                AND p.problem = a.problem
          )
    )
    SELECT
        COUNT(*) AS positive_pairs,
        COUNT(m.problem_id) AS pairs_with_metadata,
        COUNT(m.problem_rating) AS pairs_with_rating,
        COUNT(
            CASE
                WHEN m.tags IS NOT NULL AND TRIM(m.tags) != ''
                THEN 1
            END
        ) AS pairs_with_tags
    FROM future_new_solutions AS f
    LEFT JOIN codeforces_api_problems AS m
      ON f.problem = m.problem_id
    """,
    conn
)

display(positive_metadata_check)

,positive_pairs,pairs_with_metadata,pairs_with_rating,pairs_with_tags
0,943655,827488,826710,827488


In [23]:
tag_samples = pd.read_sql_query(
    """
    SELECT problem_id, problem_rating, tags
    FROM codeforces_api_problems
    WHERE tags IS NOT NULL
      AND TRIM(tags) != ''
    LIMIT 10
    """,
    conn
)

display(tag_samples)

,problem_id,problem_rating,tags
0,2275H,None,"[""combinatorics"", ""dp"", ""math""]"
1,2275G,None,"[""binary search"", ""data structures"", ""dsu"", ""g..."
2,2275F,None,"[""hashing"", ""math"", ""number theory""]"
3,2275E,None,"[""dp"", ""implementation""]"
4,2275D,None,"[""binary search"", ""brute force"", ""greedy"", ""im..."
5,2275C,None,"[""brute force"", ""data structures""]"
6,2275B,None,"[""data structures"", ""implementation""]"
7,2275A,None,"[""geometry"", ""implementation""]"
8,2274B,None,"[""greedy""]"
9,2274A,None,"[""implementation"", ""math""]"


In [24]:
# Check actual rating and tag availability
display(pd.read_sql_query(
    """
    SELECT
        COUNT(*) AS total_problems,
        COUNT(problem_rating) AS with_rating,
        SUM(CASE WHEN problem_rating IS NULL THEN 1 ELSE 0 END)
            AS missing_rating,
        SUM(CASE
            WHEN tags IS NOT NULL AND TRIM(tags) != ''
            THEN 1 ELSE 0
        END) AS with_tags
    FROM codeforces_api_problems
    """,
    conn
))

# Check what accepted_at represents
display(pd.read_sql_query(
    """
    SELECT
        COUNT(*) AS matched_pairs,
        SUM(CASE
            WHEN a.accepted_at = f.first_timestamp THEN 1 ELSE 0
        END) AS matches_first_timestamp,
        SUM(CASE
            WHEN a.accepted_at = f.last_timestamp THEN 1 ELSE 0
        END) AS matches_last_timestamp
    FROM accepted_problem_times AS a
    JOIN user_problem_features AS f
      ON a.user = f.user AND a.problem = f.problem
    """,
    conn
))

,total_problems,with_rating,missing_rating,with_tags
0,11450,11152,298,11450


,matched_pairs,matches_first_timestamp,matches_last_timestamp
0,7265808,4385965,6745824


In [25]:
accepted_table_check = pd.read_sql_query(
    """
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT user || '|' || problem) AS unique_user_problem_pairs,
        COUNT(*) - COUNT(DISTINCT user || '|' || problem)
            AS extra_rows_for_repeated_pairs,
        MIN(accepted_at) AS earliest_accepted_at,
        MAX(accepted_at) AS latest_accepted_at
    FROM accepted_problem_times
    """,
    conn
)

display(accepted_table_check)

,total_rows,unique_user_problem_pairs,extra_rows_for_repeated_pairs,earliest_accepted_at,latest_accepted_at
0,7265808,7265808,0,1266581015,1733063062


In [26]:
# Training period boundaries
TRAIN_CUTOFF = 1688169600  # July 1, 2023
TRAIN_END = 1704067200     # January 1, 2024

# Remove the temporary table if you run this cell again
conn.execute("DROP TABLE IF EXISTS selected_training_users")

# Select 300 eligible users randomly
conn.execute(
    """
    CREATE TEMP TABLE selected_training_users AS
    WITH prior_users AS (
        SELECT DISTINCT user
        FROM accepted_problem_times
        WHERE accepted_at < ?
    ),
    future_new_users AS (
        SELECT DISTINCT a.user
        FROM accepted_problem_times AS a
        JOIN prior_users AS p
          ON a.user = p.user
        WHERE a.accepted_at >= ?
          AND a.accepted_at < ?
          AND NOT EXISTS (
              SELECT 1
              FROM accepted_problem_times AS old
              WHERE old.user = a.user
                AND old.problem = a.problem
                AND old.accepted_at < ?
          )
    )
    SELECT user
    FROM future_new_users
    ORDER BY RANDOM()
    LIMIT 300
    """,
    (TRAIN_CUTOFF, TRAIN_CUTOFF, TRAIN_END, TRAIN_CUTOFF)
)

selected_count = pd.read_sql_query(
    """
    SELECT COUNT(*) AS selected_users
    FROM selected_training_users
    """,
    conn
)

display(selected_count)

,selected_users
0,300


In [27]:
table_definition = pd.read_sql_query(
    """
    SELECT sql
    FROM sqlite_master
    WHERE type = 'table'
      AND name = 'accepted_problem_times'
    """,
    conn
)

print(table_definition.iloc[0]["sql"])

CREATE TABLE accepted_problem_times (
            user TEXT NOT NULL,
            problem TEXT NOT NULL,
            accepted_at INTEGER NOT NULL,
            PRIMARY KEY (user, problem)
        )


In [28]:
from pathlib import Path

ml_dir = Path.cwd()

if ml_dir.name != "ml":
    ml_dir = ml_dir / "backend" / "ml"

print("Searching project scripts in:", ml_dir)

for file in ml_dir.glob("*.py"):
    source = file.read_text(encoding="utf-8", errors="ignore")

    if "accepted_problem_times" in source:
        print(f"\n--- Found in: {file.name} ---")

        lines = source.splitlines()

        for i, line in enumerate(lines):
            if "accepted_problem_times" in line:
                start = max(0, i - 5)
                end = min(len(lines), i + 10)

                print("\n".join(
                    f"{j + 1}: {lines[j]}"
                    for j in range(start, end)
                ))

Searching project scripts in: C:\Users\GARIMA YADUVANSHI\OneDrive\Desktop\DSAForge\DSAForge\backend\ml

--- Found in: evaluate_temporal_baseline.py ---
23: 
24: def build_accepted_times(connection):
25:     print("Building earliest accepted timestamps...")
26: 
27:     connection.execute("""
28:         CREATE TABLE IF NOT EXISTS accepted_problem_times (
29:             user TEXT NOT NULL,
30:             problem TEXT NOT NULL,
31:             accepted_at INTEGER NOT NULL,
32:             PRIMARY KEY (user, problem)
33:         )
34:     """)
35: 
36:     # Rebuild only this derived table.
37:     connection.execute("DELETE FROM accepted_problem_times")
32:             PRIMARY KEY (user, problem)
33:         )
34:     """)
35: 
36:     # Rebuild only this derived table.
37:     connection.execute("DELETE FROM accepted_problem_times")
38:     connection.commit()
39: 
40:     batch = []
41:     scanned = 0
42: 
43:     with open(CSV_PATH, "r", encoding="utf-8", newline="") as file:
44:  

In [29]:

import json
import random
import pandas as pd
import numpy as np
from collections import Counter, defaultdict

# Reproducible sampling: rerunning gives the same negative samples.
rng = random.Random(42)

# Cutoff: July 1, 2023 through January 1, 2024 (end exclusive).
TRAIN_CUTOFF = 1688169600
TRAIN_END = 1704067200

# 1. Load each selected user's accepted problems BEFORE the cutoff.
history_df = pd.read_sql_query(
    """
    SELECT a.user, a.problem, p.problem_rating, p.tags
    FROM accepted_problem_times AS a
    JOIN selected_training_users AS s ON s.user = a.user
    LEFT JOIN codeforces_api_problems AS p
        ON p.problem_id = a.problem
    WHERE a.accepted_at < ?
    """,
    conn,
    params=(TRAIN_CUTOFF,)
)

# 2. Load problems first accepted during the training window.
positive_df = pd.read_sql_query(
    """
    SELECT a.user, a.problem, p.problem_rating, p.tags
    FROM accepted_problem_times AS a
    JOIN selected_training_users AS s ON s.user = a.user
    JOIN codeforces_api_problems AS p
        ON p.problem_id = a.problem
    WHERE a.accepted_at >= ?
      AND a.accepted_at < ?
      AND p.problem_rating IS NOT NULL
      AND p.tags IS NOT NULL
    """,
    conn,
    params=(TRAIN_CUTOFF, TRAIN_END)
)

# 3. Identify every problem each user had already attempted before cutoff.
attempted_df = pd.read_sql_query(
    """
    SELECT f.user, f.problem
    FROM user_problem_features AS f
    JOIN selected_training_users AS s ON s.user = f.user
    WHERE f.first_timestamp < ?
    """,
    conn,
    params=(TRAIN_CUTOFF,)
)

# 4. Load the eligible recommendation catalogue.
catalog_df = pd.read_sql_query(
    """
    SELECT problem_id, problem_rating, tags
    FROM codeforces_api_problems
    WHERE problem_rating IS NOT NULL
      AND tags IS NOT NULL
    """,
    conn
)

def parse_tags(value):
    try:
        parsed = json.loads(value)
        if isinstance(parsed, list):
            return set(tag for tag in parsed if isinstance(tag, str))
    except (TypeError, ValueError):
        pass
    return set()

catalog = {}
for row in catalog_df.itertuples(index=False):
    tags = parse_tags(row.tags)
    if tags:
        catalog[row.problem_id] = {
            "rating": float(row.problem_rating),
            "tags": tags
        }

# Build a historical profile using only accepted problems before cutoff.
history = defaultdict(list)
user_tags = defaultdict(Counter)

for row in history_df.itertuples(index=False):
    history[row.user].append(row)

    for tag in parse_tags(row.tags):
        user_tags[row.user][tag] += 1

attempted = defaultdict(set)
for row in attempted_df.itertuples(index=False):
    attempted[row.user].add(row.problem)

positives = defaultdict(set)
for row in positive_df.itertuples(index=False):
    if row.problem in catalog:
        positives[row.user].add(row.problem)

# 5. Build ranking examples: all positives + sampled negatives per user.
records = []
groups = []

for user in history:
    positive_problems = positives[user]

    # Users without eligible future positives cannot teach the ranker
    # which candidates should be ranked highly.
    if not positive_problems:
        continue

    historical_ratings = [
        float(row.problem_rating)
        for row in history[user]
        if pd.notna(row.problem_rating)
    ]

    if not historical_ratings:
        continue

    median_rating = float(np.median(historical_ratings))
    solved_count = len(history[user])
    familiar = user_tags[user]

    # Never label a future positive as a negative.
    # Exclude previously attempted problems from the negative pool.
    blocked = attempted[user] | positive_problems

    negative_pool = [
        problem_id
        for problem_id in catalog
        if problem_id not in blocked
    ]

    negative_count = min(
        len(negative_pool),
        max(100, min(300, len(positive_problems) * 3))
    )
    negative_problems = rng.sample(negative_pool, negative_count)

    user_examples = []

    for problem_id, label in (
        [(p, 1) for p in positive_problems]
        + [(p, 0) for p in negative_problems]
    ):
        problem = catalog[problem_id]
        problem_tags = problem["tags"]
        overlap = sum(1 for tag in problem_tags if familiar[tag] > 0)

        user_examples.append({
            "user": user,
            "problem": problem_id,
            "label": label,
            "problem_rating": problem["rating"],
            "rating_gap": problem["rating"] - median_rating,
            "absolute_rating_gap": abs(problem["rating"] - median_rating),
            "historical_solved_count": solved_count,
            "problem_tag_count": len(problem_tags),
            "familiar_tag_overlap": overlap,
            "familiar_tag_ratio": overlap / len(problem_tags)
        })

    if user_examples:
        records.extend(user_examples)
        groups.append(len(user_examples))

training_df = pd.DataFrame(records)

print("Training dataset created.")
print("Selected users:", len(history))
print("Users with usable training examples:", len(groups))
print("Total training rows:", len(training_df))
print("Positive examples:", int(training_df["label"].sum()) if not training_df.empty else 0)
print("Negative examples:", int((training_df["label"] == 0).sum()) if not training_df.empty else 0)
print("Feature columns:", list(training_df.columns))
print("Ranking groups:", len(groups))


Training dataset created.
Selected users: 300
Users with usable training examples: 300
Total training rows: 91508
Positive examples: 31006
Negative examples: 60502
Feature columns: ['user', 'problem', 'label', 'problem_rating', 'rating_gap', 'absolute_rating_gap', 'historical_solved_count', 'problem_tag_count', 'familiar_tag_overlap', 'familiar_tag_ratio']
Ranking groups: 300


In [30]:

import lightgbm as lgb
import pandas as pd
import numpy as np
import random

from sklearn.model_selection import train_test_split
from sklearn.metrics import ndcg_score

# 1. Define the model features.
feature_cols = [
    "problem_rating",
    "rating_gap",
    "absolute_rating_gap",
    "historical_solved_count",
    "problem_tag_count",
    "familiar_tag_overlap",
    "familiar_tag_ratio"
]

# 2. Split by USER, never by individual problem rows.
all_users = training_df["user"].unique()

train_users, test_users = train_test_split(
    all_users,
    test_size=0.20,
    random_state=42
)

train_df = training_df[training_df["user"].isin(train_users)].copy()
test_df = training_df[training_df["user"].isin(test_users)].copy()

# Keep each user's candidate problems together.
train_df = train_df.sort_values("user", kind="stable").reset_index(drop=True)
test_df = test_df.sort_values("user", kind="stable").reset_index(drop=True)

X_train = train_df[feature_cols]
y_train = train_df["label"]

X_test = test_df[feature_cols]
y_test = test_df["label"]

# Number of candidate problems in each user's ranking group.
train_groups = train_df.groupby("user", sort=False).size().tolist()
test_groups = test_df.groupby("user", sort=False).size().tolist()

print("Training users:", len(train_users))
print("Testing users:", len(test_users))
print("Training rows:", len(train_df))
print("Testing rows:", len(test_df))

# 3. Train a learning-to-rank model.
model = lgb.LGBMRanker(
    objective="lambdarank",
    metric="ndcg",
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=30,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=42,
    verbosity=-1
)

model.fit(
    X_train,
    y_train,
    group=train_groups,
    eval_set=[(X_test, y_test)],
    eval_group=[test_groups],
    eval_at=[10, 50],
    callbacks=[
        lgb.early_stopping(50),
        lgb.log_evaluation(25)
    ]
)

print("\nTraining completed.")
print("Best iteration:", model.best_iteration_)


Training users: 240
Testing users: 60
Training rows: 74396
Testing rows: 17112
Training until validation scores don't improve for 50 rounds


C:\Users\GARIMA YADUVANSHI\OneDrive\Desktop\DSAForge\DSAForge\backend\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[25]	valid_0's ndcg@10: 0.64123	valid_0's ndcg@50: 0.643853
[50]	valid_0's ndcg@10: 0.630087	valid_0's ndcg@50: 0.645408
Early stopping, best iteration is:
[1]	valid_0's ndcg@10: 0.761882	valid_0's ndcg@50: 0.719321

Training completed.
Best iteration: 1


In [32]:

test_df["predicted_score"] = model.predict(
    test_df[feature_cols],
    num_iteration=model.best_iteration_
)

print("Predictions created:", "predicted_score" in test_df.columns)
print(test_df[["user", "problem", "label", "predicted_score"]].head())


Predictions created: True
        user problem  label  predicted_score
0  user10034   1900A      1         0.048151
1  user10034   1915B      1         0.048151
2  user10034   1873A      1         0.054362
3  user10034   1915C      1         0.054362
4  user10034   1873E      1         0.026850


In [33]:

from sklearn.metrics import ndcg_score
import numpy as np
import pandas as pd

results = {
    "LightGBM": [],
    "Difficulty-only": [],
    "Topic-only": [],
    "Hybrid": [],
}
random_scores = []

rng = np.random.default_rng(42)

for user, group in test_df.groupby("user", sort=False):
    group = group.copy()
    labels = group["label"].to_numpy()

    if labels.sum() == 0:
        continue

    # Model ranking
    model_scores = group["predicted_score"].to_numpy()

    # Baseline 1: closest to historical median difficulty
    difficulty_scores = -group["absolute_rating_gap"].to_numpy()

    # Baseline 2: familiar topic overlap
    topic_scores = group["familiar_tag_ratio"].to_numpy()

    # Hybrid: combine within-user percentile ranks
    difficulty_rank = pd.Series(difficulty_scores).rank(pct=True).to_numpy()
    topic_rank = pd.Series(topic_scores).rank(pct=True).to_numpy()
    hybrid_scores = 0.5 * difficulty_rank + 0.5 * topic_rank

    for name, scores in [
        ("LightGBM", model_scores),
        ("Difficulty-only", difficulty_scores),
        ("Topic-only", topic_scores),
        ("Hybrid", hybrid_scores),
    ]:
        results[name].append(
            ndcg_score([labels], [scores], k=10)
        )

    # Average random-ranking NDCG over 20 random orderings.
    random_scores.extend([
        ndcg_score([labels], [rng.random(len(labels))], k=10)
        for _ in range(20)
    ])

summary = pd.DataFrame([
    {
        "Strategy": name,
        "Mean NDCG@10": np.mean(values),
        "Users evaluated": len(values)
    }
    for name, values in results.items()
])

summary.loc[len(summary)] = {
    "Strategy": "Random (20 trials/user)",
    "Mean NDCG@10": np.mean(random_scores),
    "Users evaluated": len(results["LightGBM"])
}

print(summary.sort_values("Mean NDCG@10", ascending=False).to_string(index=False))


               Strategy  Mean NDCG@10  Users evaluated
               LightGBM      0.432852               60
                 Hybrid      0.378619               60
        Difficulty-only      0.374445               60
             Topic-only      0.282133               60
Random (20 trials/user)      0.250874               60


In [34]:

# Basic integrity checks for the current training dataset

print("1. Missing feature values:")
print(training_df[feature_cols].isna().sum())

print("\n2. Label distribution:")
print(training_df["label"].value_counts())

print("\n3. Duplicate user-problem pairs:")
print(training_df.duplicated(["user", "problem"]).sum())

print("\n4. Check that every user's group contains positive examples:")
user_label_check = training_df.groupby("user")["label"].agg(
    positive_count="sum",
    candidate_count="count"
)
print(user_label_check.describe())

print("\n5. Check train/test user overlap:")
print("Overlapping users:", len(set(train_users) & set(test_users)))

print("\n6. Check whether any training users appear in test data:")
print("Training users:", len(set(train_df["user"])))
print("Testing users:", len(set(test_df["user"])))


1. Missing feature values:
problem_rating             0
rating_gap                 0
absolute_rating_gap        0
historical_solved_count    0
problem_tag_count          0
familiar_tag_overlap       0
familiar_tag_ratio         0
dtype: int64

2. Label distribution:
label
0    60502
1    31006
Name: count, dtype: int64

3. Duplicate user-problem pairs:
0

4. Check that every user's group contains positive examples:
       positive_count  candidate_count
count      300.000000       300.000000
mean       103.353333       305.026667
std        112.309870       189.149582
min          1.000000       101.000000
25%         22.000000       122.000000
50%         68.000000       272.000000
75%        142.000000       442.000000
max        592.000000       892.000000

5. Check train/test user overlap:
Overlapping users: 0

6. Check whether any training users appear in test data:
Training users: 240
Testing users: 60


In [35]:

# Inspect one user's history and training labels.
# This is read-only; it does not change your model or database.

sample_user = training_df["user"].iloc[0]

print("Sample user:", sample_user)
print("Training cutoff:", pd.to_datetime(
    TRAIN_CUTOFF, unit="s", utc=True
))
print("Training period ends:", pd.to_datetime(
    TRAIN_END, unit="s", utc=True
))

# Problems accepted before the cutoff
past = pd.read_sql_query(
    """
    SELECT problem, accepted_at
    FROM accepted_problem_times
    WHERE user = ? AND accepted_at < ?
    ORDER BY accepted_at
    LIMIT 10
    """,
    conn,
    params=(sample_user, TRAIN_CUTOFF)
)

# Problems first accepted in the training window
future = pd.read_sql_query(
    """
    SELECT problem, accepted_at
    FROM accepted_problem_times
    WHERE user = ?
      AND accepted_at >= ?
      AND accepted_at < ?
    ORDER BY accepted_at
    LIMIT 10
    """,
    conn,
    params=(sample_user, TRAIN_CUTOFF, TRAIN_END)
)

# Positive examples actually included in our training dataset
labels = training_df[
    (training_df["user"] == sample_user)
    & (training_df["label"] == 1)
][["problem", "label"]].head(10)

print("\nHistorical accepted problems:")
print(past)

print("\nProblems first accepted during training window:")
print(future)

print("\nPositive labels used by the model:")
print(labels)


Sample user: user13004
Training cutoff: 2023-07-01 00:00:00+00:00
Training period ends: 2024-01-01 00:00:00+00:00

Historical accepted problems:
  problem  accepted_at
0    158A   1628441834
1     50A   1628508092
2    282A   1628579100
3    263A   1628584375
4   1555A   1628939345
5   1560A   1629299189
6   1560B   1629301631
7   1560C   1629304418
8   1561A   1629817696
9   1561B   1629893993

Problems first accepted during training window:
  problem  accepted_at
0   1528D   1688546148
1    515E   1689074851
2   1844A   1689086192
3   1844B   1689086484
4   1844C   1689087057
5   1844D   1689088135
6   1844E   1689094578
7  1854A2   1690643371
8  1854A1   1690643389
9   1854B   1690648457

Positive labels used by the model:
  problem  label
0   1119E      1
1    150E      1
2    914F      1
3    484E      1
4   1178G      1
5   1909B      1
6  1868B2      1
7   1889B      1
8    455E      1
9    437D      1


In [36]:

import sqlite3
import pandas as pd
import numpy as np
import json
import random
from collections import defaultdict, Counter

# Unix timestamps (UTC), end dates are exclusive.
HISTORY_END = 1672531200   # Jan 1, 2023
TRAIN_END = 1688169600     # Jul 1, 2023
TEST_END = 1704067200      # Jan 1, 2024

rng = random.Random(42)

# 1. Select users who have both historical activity and later accepted problems.
conn.execute("DROP TABLE IF EXISTS temp.temporal_users")

conn.execute(
    """
    CREATE TEMP TABLE temporal_users AS
    SELECT user
    FROM accepted_problem_times
    GROUP BY user
    HAVING SUM(CASE WHEN accepted_at < ? THEN 1 ELSE 0 END) > 0
       AND SUM(CASE WHEN accepted_at >= ? AND accepted_at < ? THEN 1 ELSE 0 END) > 0
    ORDER BY RANDOM()
    LIMIT 300
    """,
    (HISTORY_END, TRAIN_END, TEST_END)
)

print(
    "Selected temporal users:",
    conn.execute("SELECT COUNT(*) FROM temporal_users").fetchone()[0]
)

# 2. Load only historical accepted problems for these users.
history_df = pd.read_sql_query(
    """
    SELECT a.user, a.problem, p.problem_rating, p.tags
    FROM accepted_problem_times a
    JOIN temporal_users u ON u.user = a.user
    LEFT JOIN codeforces_api_problems p ON p.problem_id = a.problem
    WHERE a.accepted_at < ?
    """,
    conn,
    params=(HISTORY_END,)
)

# 3. Load problems first accepted during Jan-Jun 2023.
train_labels_df = pd.read_sql_query(
    """
    SELECT a.user, a.problem, p.problem_rating, p.tags
    FROM accepted_problem_times a
    JOIN temporal_users u ON u.user = a.user
    JOIN codeforces_api_problems p ON p.problem_id = a.problem
    WHERE a.accepted_at >= ?
      AND a.accepted_at < ?
      AND p.problem_rating IS NOT NULL
      AND p.tags IS NOT NULL
    """,
    conn,
    params=(HISTORY_END, TRAIN_END)
)

# 4. Load problems first accepted during Jul-Dec 2023.
test_labels_df = pd.read_sql_query(
    """
    SELECT a.user, a.problem, p.problem_rating, p.tags
    FROM accepted_problem_times a
    JOIN temporal_users u ON u.user = a.user
    JOIN codeforces_api_problems p ON p.problem_id = a.problem
    WHERE a.accepted_at >= ?
      AND a.accepted_at < ?
      AND p.problem_rating IS NOT NULL
      AND p.tags IS NOT NULL
    """,
    conn,
    params=(TRAIN_END, TEST_END)
)

# 5. Load the eligible problem catalogue.
catalog_df = pd.read_sql_query(
    """
    SELECT problem_id, problem_rating, tags
    FROM codeforces_api_problems
    WHERE problem_rating IS NOT NULL AND tags IS NOT NULL
    """,
    conn
)

def parse_tags(value):
    try:
        parsed = json.loads(value)
        return set(x for x in parsed if isinstance(x, str)) if isinstance(parsed, list) else set()
    except (ValueError, TypeError):
        return set()

catalog = {}
for row in catalog_df.itertuples(index=False):
    tags = parse_tags(row.tags)
    if tags:
        catalog[row.problem_id] = {
            "rating": float(row.problem_rating),
            "tags": tags
        }

# 6. Historical user profiles: historical ratings and familiar tags only.
history = defaultdict(list)
user_tags = defaultdict(Counter)

for row in history_df.itertuples(index=False):
    history[row.user].append(row)
    for tag in parse_tags(row.tags):
        user_tags[row.user][tag] += 1

train_positives = defaultdict(set)
for row in train_labels_df.itertuples(index=False):
    if row.problem in catalog:
        train_positives[row.user].add(row.problem)

test_positives = defaultdict(set)
for row in test_labels_df.itertuples(index=False):
    if row.problem in catalog:
        test_positives[row.user].add(row.problem)

# 7. Generate candidate rows.
# For this first temporal experiment, use positives and sampled negatives.
def build_examples(users, positive_map, historical_map, tag_map, catalogue, rng):
    examples = []

    for user in users:
        if user not in historical_map or not positive_map[user]:
            continue

        past_ratings = [
            float(row.problem_rating)
            for row in historical_map[user]
            if pd.notna(row.problem_rating)
        ]
        if not past_ratings:
            continue

        median_rating = float(np.median(past_ratings))
        familiar = tag_map[user]
        past_problems = {row.problem for row in historical_map[user]}
        positives = positive_map[user]

        blocked = past_problems | positives
        negative_pool = [p for p in catalogue if p not in blocked]
        n_negatives = min(len(negative_pool), max(100, 3 * len(positives)))
        negatives = rng.sample(negative_pool, n_negatives)

        for problem_id, label in (
            [(p, 1) for p in positives] +
            [(p, 0) for p in negatives]
        ):
            problem = catalogue[problem_id]
            tags = problem["tags"]
            overlap = sum(1 for tag in tags if familiar[tag] > 0)

            examples.append({
                "user": user,
                "problem": problem_id,
                "label": label,
                "problem_rating": problem["rating"],
                "rating_gap": problem["rating"] - median_rating,
                "absolute_rating_gap": abs(problem["rating"] - median_rating),
                "historical_solved_count": len(historical_map[user]),
                "problem_tag_count": len(tags),
                "familiar_tag_overlap": overlap,
                "familiar_tag_ratio": overlap / len(tags)
            })

    return pd.DataFrame(examples)

temporal_train_df = build_examples(
    list(train_positives.keys()),
    train_positives, history, user_tags, catalog, rng
)

temporal_test_df = build_examples(
    list(test_positives.keys()),
    test_positives, history, user_tags, catalog, rng
)

print("\nTemporal dataset summary")
print("Training rows:", len(temporal_train_df))
print("Training positive labels:", int(temporal_train_df["label"].sum()) if not temporal_train_df.empty else 0)
print("Testing rows:", len(temporal_test_df))
print("Testing positive labels:", int(temporal_test_df["label"].sum()) if not temporal_test_df.empty else 0)
print("Training users:", temporal_train_df["user"].nunique() if not temporal_train_df.empty else 0)
print("Testing users:", temporal_test_df["user"].nunique() if not temporal_test_df.empty else 0)


Selected temporal users: 300

Temporal dataset summary
Training rows: 123018
Training positive labels: 29450
Testing rows: 139737
Testing positive labels: 33745
Training users: 270
Testing users: 295


In [37]:

from sklearn.metrics import ndcg_score
import lightgbm as lgb
import numpy as np
import pandas as pd

feature_cols = [
    "problem_rating",
    "rating_gap",
    "absolute_rating_gap",
    "historical_solved_count",
    "problem_tag_count",
    "familiar_tag_overlap",
    "familiar_tag_ratio"
]

# Use only users represented in both datasets.
train_user_set = set(temporal_train_df["user"].unique())
test_user_set = set(temporal_test_df["user"].unique())
common_users = sorted(train_user_set & test_user_set)

train_df = temporal_train_df[
    temporal_train_df["user"].isin(common_users)
].sort_values("user", kind="stable").reset_index(drop=True)

test_df = temporal_test_df[
    temporal_test_df["user"].isin(common_users)
].sort_values("user", kind="stable").reset_index(drop=True)

X_train = train_df[feature_cols]
y_train = train_df["label"]
X_test = test_df[feature_cols]
y_test = test_df["label"]

train_groups = train_df.groupby("user", sort=False).size().tolist()
test_groups = test_df.groupby("user", sort=False).size().tolist()

print("Common users:", len(common_users))
print("Training rows:", len(train_df))
print("Testing rows:", len(test_df))

temporal_model = lgb.LGBMRanker(
    objective="lambdarank",
    metric="ndcg",
    n_estimators=300,
    learning_rate=0.03,
    num_leaves=15,
    min_child_samples=50,
    reg_lambda=2.0,
    random_state=42,
    verbosity=-1
)

temporal_model.fit(
    X_train,
    y_train,
    group=train_groups
)

# Score the later-period candidates.
test_df["predicted_score"] = temporal_model.predict(X_test)

# Evaluate each user separately.
ndcg_values = []
recall_values = []

for user, group in test_df.groupby("user", sort=False):
    group = group.sort_values("predicted_score", ascending=False)
    labels = group["label"].to_numpy()

    ndcg_values.append(
        ndcg_score([labels], [group["predicted_score"].to_numpy()], k=10)
    )

    positives = int(labels.sum())
    recall_values.append(
        group.head(10)["label"].sum() / positives if positives else 0.0
    )

print("\nTemporal evaluation")
print(f"Users evaluated: {len(ndcg_values)}")
print(f"Mean NDCG@10: {np.mean(ndcg_values):.4f}")
print(f"Mean Recall@10: {np.mean(recall_values):.4f}")

print("\nFeature importance:")
print(
    pd.Series(
        temporal_model.feature_importances_,
        index=feature_cols
    ).sort_values(ascending=False)
)


Common users: 270
Training rows: 123018
Testing rows: 135659

Temporal evaluation
Users evaluated: 270
Mean NDCG@10: 0.4414
Mean Recall@10: 0.0884

Feature importance:
historical_solved_count    1018
problem_rating              978
rating_gap                  614
problem_tag_count           539
familiar_tag_overlap        428
absolute_rating_gap         356
familiar_tag_ratio          267
dtype: int32


In [38]:

from sklearn.metrics import ndcg_score
import numpy as np
import pandas as pd

baseline_results = {
    "LightGBM": [],
    "Difficulty-only": [],
    "Topic-only": [],
    "Hybrid": [],
}
random_results = []
rng = np.random.default_rng(42)

for user, group in test_df.groupby("user", sort=False):
    group = group.copy()
    labels = group["label"].to_numpy()

    model_scores = group["predicted_score"].to_numpy()
    difficulty_scores = -group["absolute_rating_gap"].to_numpy()
    topic_scores = group["familiar_tag_ratio"].to_numpy()

    difficulty_rank = pd.Series(difficulty_scores).rank(pct=True).to_numpy()
    topic_rank = pd.Series(topic_scores).rank(pct=True).to_numpy()
    hybrid_scores = 0.5 * difficulty_rank + 0.5 * topic_rank

    for name, scores in [
        ("LightGBM", model_scores),
        ("Difficulty-only", difficulty_scores),
        ("Topic-only", topic_scores),
        ("Hybrid", hybrid_scores),
    ]:
        baseline_results[name].append(
            ndcg_score([labels], [scores], k=10)
        )

    for _ in range(20):
        random_scores = rng.random(len(group))
        random_results.append(
            ndcg_score([labels], [random_scores], k=10)
        )

summary = pd.DataFrame([
    {
        "Strategy": name,
        "Mean NDCG@10": np.mean(values)
    }
    for name, values in baseline_results.items()
])

summary.loc[len(summary)] = {
    "Strategy": "Random (20 trials/user)",
    "Mean NDCG@10": np.mean(random_results)
}

print(summary.sort_values("Mean NDCG@10", ascending=False).to_string(index=False))


               Strategy  Mean NDCG@10
               LightGBM      0.441369
        Difficulty-only      0.345542
                 Hybrid      0.341105
             Topic-only      0.248401
Random (20 trials/user)      0.222241


In [39]:

from pathlib import Path
import json
import lightgbm as lgb

MODEL_DIR = PROJECT_ROOT / "backend" / "ml" / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

model_path = MODEL_DIR / "dsaforge_ranker.txt"
features_path = MODEL_DIR / "feature_columns.json"

# Save the trained LightGBM model.
temporal_model.booster_.save_model(str(model_path))

# Save the exact feature order required during prediction.
with open(features_path, "w", encoding="utf-8") as file:
    json.dump(feature_cols, file, indent=2)

print("Model saved:", model_path)
print("Feature list saved:", features_path)
print("Model file exists:", model_path.exists())
print("Feature file exists:", features_path.exists())
print("Best iteration:", temporal_model.best_iteration_)


Model saved: C:\Users\GARIMA YADUVANSHI\OneDrive\Desktop\DSAForge\DSAForge\backend\ml\models\dsaforge_ranker.txt
Feature list saved: C:\Users\GARIMA YADUVANSHI\OneDrive\Desktop\DSAForge\DSAForge\backend\ml\models\feature_columns.json
Model file exists: True
Feature file exists: True
Best iteration: 0


In [40]:

import json
import lightgbm as lgb
import pandas as pd
import numpy as np

# Reload the exact model saved to disk.
loaded_model = lgb.Booster(
    model_file=str(model_path)
)

with open(features_path, "r", encoding="utf-8") as file:
    loaded_features = json.load(file)

print("Features loaded:", loaded_features)
print("Model trees:", loaded_model.num_trees())
print("Model feature count:", loaded_model.num_feature())

# Generate predictions on the temporal test candidates.
test_features = test_df[loaded_features]
predictions = loaded_model.predict(test_features)

print("\nPrediction count:", len(predictions))
print("All predictions finite:", np.isfinite(predictions).all())
print("Unique prediction scores:", len(np.unique(predictions)))
print("First 10 scores:", predictions[:10])


Features loaded: ['problem_rating', 'rating_gap', 'absolute_rating_gap', 'historical_solved_count', 'problem_tag_count', 'familiar_tag_overlap', 'familiar_tag_ratio']
Model trees: 300
Model feature count: 7

Prediction count: 135659
All predictions finite: True
Unique prediction scores: 18190
First 10 scores: [-1.97213251 -1.48774548 -1.32878972 -1.26037227  0.8084919   0.8084919
 -2.44182523 -1.70834371  0.04497885 -1.26875162]


In [41]:
# Select one user from the test data
sample_user = test_df["user"].iloc[0]

# Get this user's candidate problems
user_candidates = test_df[test_df["user"] == sample_user].copy()

# Rank problems using the saved model
user_candidates["predicted_score"] = loaded_model.predict(
    user_candidates[loaded_features]
)

# Display the top 10 recommendations
top_10 = user_candidates.sort_values(
    "predicted_score",
    ascending=False
).head(10)

display(
    top_10[
        [
            "problem",
            "problem_rating",
            "predicted_score",
            "label"
        ]
    ].reset_index(drop=True)
)

,problem,problem_rating,predicted_score,label
0,1884A,800.0,0.808492,1
1,148A,800.0,0.808492,0
2,1898A,800.0,0.808492,1
3,1539B,800.0,0.808492,0
4,1915B,800.0,0.808492,1
5,1779A,800.0,0.808492,0
6,1899A,800.0,0.808492,1
7,1796A,800.0,0.808492,0
8,381A,800.0,0.808492,1
9,1846B,800.0,0.808492,1


In [42]:
display(
    top_10[
        [
            "problem",
            "problem_rating",
            "rating_gap",
            "absolute_rating_gap",
            "historical_solved_count",
            "problem_tag_count",
            "familiar_tag_overlap",
            "familiar_tag_ratio",
            "predicted_score",
            "label"
        ]
    ].reset_index(drop=True)
)

,problem,problem_rating,rating_gap,absolute_rating_gap,historical_solved_count,problem_tag_count,familiar_tag_overlap,familiar_tag_ratio,predicted_score,label
0,1884A,800.0,0.0,0.0,108,3,3,1.0,0.808492,1
1,148A,800.0,0.0,0.0,108,3,3,1.0,0.808492,0
2,1898A,800.0,0.0,0.0,108,3,3,1.0,0.808492,1
3,1539B,800.0,0.0,0.0,108,3,3,1.0,0.808492,0
4,1915B,800.0,0.0,0.0,108,3,3,1.0,0.808492,1
5,1779A,800.0,0.0,0.0,108,3,3,1.0,0.808492,0
6,1899A,800.0,0.0,0.0,108,3,3,1.0,0.808492,1
7,1796A,800.0,0.0,0.0,108,3,3,1.0,0.808492,0
8,381A,800.0,0.0,0.0,108,3,3,1.0,0.808492,1
9,1846B,800.0,0.0,0.0,108,3,3,1.0,0.808492,1


In [43]:
import json
import pandas as pd

problem_ids = top_10["problem"].astype(str).tolist()
placeholders = ",".join(["?"] * len(problem_ids))

tags_check = pd.read_sql_query(
    f"""
    SELECT problem_id, problem_name, problem_rating, tags
    FROM codeforces_api_problems
    WHERE problem_id IN ({placeholders})
    """,
    conn,
    params=problem_ids
)

tags_check["parsed_tags"] = tags_check["tags"].apply(
    lambda x: json.loads(x) if isinstance(x, str) else []
)

display(tags_check[[
    "problem_id",
    "problem_name",
    "problem_rating",
    "parsed_tags"
]])

,problem_id,problem_name,problem_rating,parsed_tags
0,148A,Insomnia cure,800.0,"[constructive algorithms, implementation, math]"
1,1539B,Love Song,800.0,"[dp, implementation, strings]"
2,1779A,Hall of Fame,800.0,"[constructive algorithms, greedy, strings]"
3,1796A,Typical Interview Problem,800.0,"[brute force, implementation, strings]"
4,1846B,Rudolph and Tic-Tac-Toe,800.0,"[brute force, implementation, strings]"
5,1884A,Simple Design,800.0,"[brute force, greedy, math]"
6,1898A,Milica and String,800.0,"[brute force, implementation, strings]"
7,1899A,Game with Integers,800.0,"[games, math, number theory]"
8,1915B,Not Quite Latin Square,800.0,"[bitmasks, brute force, implementation]"
9,381A,Sereja and Dima,800.0,"[greedy, implementation, two pointers]"


In [44]:
import inspect

print(inspect.getsource(build_examples))

def build_examples(users, positive_map, historical_map, tag_map, catalogue, rng):
    examples = []

    for user in users:
        if user not in historical_map or not positive_map[user]:
            continue

        past_ratings = [
            float(row.problem_rating)
            for row in historical_map[user]
            if pd.notna(row.problem_rating)
        ]
        if not past_ratings:
            continue

        median_rating = float(np.median(past_ratings))
        familiar = tag_map[user]
        past_problems = {row.problem for row in historical_map[user]}
        positives = positive_map[user]

        blocked = past_problems | positives
        negative_pool = [p for p in catalogue if p not in blocked]
        n_negatives = min(len(negative_pool), max(100, 3 * len(positives)))
        negatives = rng.sample(negative_pool, n_negatives)

        for problem_id, label in (
            [(p, 1) for p in positives] +
            [(p, 0) for p in negatives]
        ):
    

In [49]:
overlap = sum(1 for tag in tags if familiar[tag] > 0)


In [50]:
print("catalogue exists:", "catalogue" in globals())
print("tag_map exists:", "tag_map" in globals())
print("history exists:", "history" in globals())
print("training_df exists:", "training_df" in globals())
print("test_df exists:", "test_df" in globals())
print("build_examples exists:", "build_examples" in globals())

catalogue exists: False
tag_map exists: False
history exists: True
training_df exists: True
test_df exists: True
build_examples exists: True


In [51]:
print("Database connection:", conn.execute("SELECT 1").fetchone())

print("\nMetadata columns:")
display(pd.read_sql_query(
    "PRAGMA table_info(codeforces_api_problems)",
    conn
).loc[:, ["name", "type"]])

print("\nExample problem tags:")
display(pd.read_sql_query(
    """
    SELECT problem_id, problem_rating, tags
    FROM codeforces_api_problems
    WHERE tags IS NOT NULL AND tags != ''
    LIMIT 5
    """,
    conn
))

Database connection: (1,)

Metadata columns:


,name,type
0,problem_id,TEXT
1,contest_id,INTEGER
2,problem_index,TEXT
3,problem_name,TEXT
4,problem_rating,REAL
5,tags,TEXT



Example problem tags:


,problem_id,problem_rating,tags
0,2275H,None,"[""combinatorics"", ""dp"", ""math""]"
1,2275G,None,"[""binary search"", ""data structures"", ""dsu"", ""g..."
2,2275F,None,"[""hashing"", ""math"", ""number theory""]"
3,2275E,None,"[""dp"", ""implementation""]"
4,2275D,None,"[""binary search"", ""brute force"", ""greedy"", ""im..."


In [53]:
import json
import pandas as pd

metadata_df = pd.read_sql_query(
    """
    SELECT problem_id, problem_rating, tags
    FROM codeforces_api_problems
    WHERE problem_rating IS NOT NULL
      AND tags IS NOT NULL
      AND tags != ''
    """,
    conn
)

catalogue = {}

for _, row in metadata_df.iterrows():
    try:
        tags = json.loads(row["tags"])
    except (json.JSONDecodeError, TypeError):
        continue

    if not isinstance(tags, list) or not tags:
        continue

    catalogue[row["problem_id"]] = {
        "rating": float(row["problem_rating"]),
        "tags": tags
    }

print("Problems in catalogue:", len(catalogue))
print("Example entry:", next(iter(catalogue.items())))

Problems in catalogue: 10990
Example entry: ('2269B', {'rating': 1000.0, 'tags': ['brute force', 'implementation']})


In [54]:
print("History type:", type(history))
print("Number of users in history:", len(history))

sample_user = next(iter(history))
print("\nSample user:", sample_user)
print("History entry type:", type(history[sample_user]))
print("First history entry:")

if history[sample_user]:
    print(history[sample_user][0])

History type: <class 'collections.defaultdict'>
Number of users in history: 300

Sample user: user3106
History entry type: <class 'list'>
First history entry:
Pandas(user='user3106', problem='102B', problem_rating=1000.0, tags='["implementation"]')


In [55]:
from collections import Counter, defaultdict
import json

tag_map = defaultdict(Counter)

for user, solved_problems in history.items():
    for row in solved_problems:
        if not isinstance(row.tags, str):
            continue

        try:
            tags = json.loads(row.tags)
        except (json.JSONDecodeError, TypeError):
            continue

        if isinstance(tags, list):
            tag_map[user].update(tags)

print("Users with topic history:", len(tag_map))

sample_user = next(iter(tag_map))
print("\nSample user:", sample_user)
print("Most practised topics:")
print(tag_map[sample_user].most_common(10))

Users with topic history: 296

Sample user: user3106
Most practised topics:
[('implementation', 150), ('math', 135), ('greedy', 118), ('brute force', 53), ('constructive algorithms', 46), ('sortings', 42), ('strings', 33), ('dp', 25), ('number theory', 19), ('binary search', 17)]


In [56]:
all_topics = sorted({
    tag
    for problem in catalogue.values()
    for tag in problem["tags"]
})

print("Total topics:", len(all_topics))
print("\nTopics:")
print(all_topics)

Total topics: 38

Topics:
['*special', '2-sat', 'binary search', 'bitmasks', 'brute force', 'chinese remainder theorem', 'combinatorics', 'communication', 'constructive algorithms', 'data structures', 'dfs and similar', 'divide and conquer', 'dp', 'dsu', 'expression parsing', 'fft', 'flows', 'games', 'geometry', 'graph matchings', 'graphs', 'greedy', 'hashing', 'implementation', 'interactive', 'math', 'matrices', 'meet-in-the-middle', 'number theory', 'probabilities', 'schedules', 'shortest paths', 'sortings', 'string suffix structures', 'strings', 'ternary search', 'trees', 'two pointers']


In [61]:
import numpy as np
import pandas as pd

def add_topic_features(df, tag_map, catalogue, all_topics):
    result = df.copy()

    # Get the tags for each candidate problem
    candidate_tags = result["problem"].map(
        lambda problem: set(
            catalogue.get(problem, {}).get("tags", [])
        )
    )

    for topic in all_topics:
        column = (
            "topic_" + topic
            .replace("*", "special_")
            .replace(" ", "_")
            .replace("-", "_")
        )

        result[column] = [
            tag_map.get(user, {}).get(topic, 0)
            if topic in tags else 0
            for user, tags in zip(result["user"], candidate_tags)
        ]

    return result


print("Topic feature names:")
print([
    "topic_" + t.replace(" ", "_").replace("*", "special_").replace("-", "_")
    for t in all_topics
])

Topic feature names:
['topic_special_special', 'topic_2_sat', 'topic_binary_search', 'topic_bitmasks', 'topic_brute_force', 'topic_chinese_remainder_theorem', 'topic_combinatorics', 'topic_communication', 'topic_constructive_algorithms', 'topic_data_structures', 'topic_dfs_and_similar', 'topic_divide_and_conquer', 'topic_dp', 'topic_dsu', 'topic_expression_parsing', 'topic_fft', 'topic_flows', 'topic_games', 'topic_geometry', 'topic_graph_matchings', 'topic_graphs', 'topic_greedy', 'topic_hashing', 'topic_implementation', 'topic_interactive', 'topic_math', 'topic_matrices', 'topic_meet_in_the_middle', 'topic_number_theory', 'topic_probabilities', 'topic_schedules', 'topic_shortest_paths', 'topic_sortings', 'topic_string_suffix_structures', 'topic_strings', 'topic_ternary_search', 'topic_trees', 'topic_two_pointers']


In [63]:
sample_features = add_topic_features(
    training_df.head(5),
    tag_map,
    catalogue,
    all_topics
)

topic_columns = [
    "topic_strings",
    "topic_math",
    "topic_dp",
    "topic_implementation",
    "topic_greedy"
]

display(
    sample_features[
        ["user", "problem", "label"] + topic_columns
    ]
)

,user,problem,label,topic_strings,topic_math,topic_dp,topic_implementation,topic_greedy
0,user13004,1119E,1,0,0,0,0,0
1,user13004,150E,1,0,0,0,0,0
2,user13004,914F,1,0,0,0,0,0
3,user13004,484E,1,0,0,0,0,0
4,user13004,1178G,1,0,0,0,0,0


In [64]:

print("Training dataset shape:", training_df.shape)
print("Testing dataset shape:", test_df.shape)

print("\nTraining users:", training_df["user"].nunique())
print("Testing users:", test_df["user"].nunique())

print("\nTraining columns:")
print(training_df.columns.tolist())

print("\nTesting columns:")
print(test_df.columns.tolist())


Training dataset shape: (91508, 10)
Testing dataset shape: (135659, 11)

Training users: 300
Testing users: 270

Training columns:
['user', 'problem', 'label', 'problem_rating', 'rating_gap', 'absolute_rating_gap', 'historical_solved_count', 'problem_tag_count', 'familiar_tag_overlap', 'familiar_tag_ratio']

Testing columns:
['user', 'problem', 'label', 'problem_rating', 'rating_gap', 'absolute_rating_gap', 'historical_solved_count', 'problem_tag_count', 'familiar_tag_overlap', 'familiar_tag_ratio', 'predicted_score']


In [65]:

for name in [
    "history_df",
    "train_labels_df",
    "test_labels_df",
    "history",
    "train_positives",
    "test_positives",
    "temporal_model",
    "X_train",
    "X_test",
    "train_groups",
    "test_groups"
]:
    print(f"{name}: {name in globals()}")


history_df: True
train_labels_df: True
test_labels_df: True
history: True
train_positives: True
test_positives: True
temporal_model: True
X_train: True
X_test: True
train_groups: True
test_groups: True


In [66]:

print("History users:", len(history))
print("Topic-history users:", len(tag_map))

print("\nUsers in temporal history missing from tag_map:")
print(len(set(history) - set(tag_map)))

print("\nExample temporal user:", next(iter(history)))
print("Example user in tag_map:", next(iter(tag_map)))


History users: 300
Topic-history users: 296

Users in temporal history missing from tag_map:
4

Example temporal user: user3106
Example user in tag_map: user3106


In [67]:

print("Train labels columns:")
print(train_labels_df.columns.tolist())

print("\nTest labels columns:")
print(test_labels_df.columns.tolist())

print("\nOne training label row:")
display(train_labels_df.head(1))

print("\nOne test label row:")
display(test_labels_df.head(1))


Train labels columns:
['user', 'problem', 'problem_rating', 'tags']

Test labels columns:
['user', 'problem', 'problem_rating', 'tags']

One training label row:


,user,problem,problem_rating,tags
0,user12423,1842H,3000.0,"[""bitmasks"", ""dp"", ""graphs"", ""math"", ""probabil..."



One test label row:


,user,problem,problem_rating,tags
0,user12423,1916H2,2700.0,"[""combinatorics"", ""dp"", ""math"", ""matrices"", ""s..."


In [68]:

for name in [
    "train_examples",
    "test_examples",
    "train_df",
    "temporal_train_df",
    "temporal_test_df",
    "test_df",
    "training_df",
]:
    obj = globals().get(name)
    if obj is not None:
        print(f"{name}: {obj.shape if hasattr(obj, 'shape') else type(obj).__name__}")


train_df: (123018, 10)
temporal_train_df: (123018, 10)
temporal_test_df: (139737, 10)
test_df: (135659, 11)
training_df: (91508, 10)


In [69]:

import json

def add_topic_features(df, tag_map, catalogue, all_topics):
    result = df.copy()

    candidate_tags = result["problem"].map(
        lambda problem: set(
            catalogue.get(problem, {}).get("tags", [])
        )
    )

    user_topic_counts = result["user"].map(
        lambda user: tag_map.get(user, {})
    )

    for topic in all_topics:
        column = (
            "topic_"
            + topic.replace("*", "special_")
                  .replace(" ", "_")
                  .replace("-", "_")
        )

        result[column] = [
            counts.get(topic, 0) if topic in tags else 0
            for counts, tags in zip(user_topic_counts, candidate_tags)
        ]

    return result


temporal_train_with_topics = add_topic_features(
    temporal_train_df, tag_map, catalogue, all_topics
)

temporal_test_with_topics = add_topic_features(
    temporal_test_df, tag_map, catalogue, all_topics
)

print("Training shape:", temporal_train_with_topics.shape)
print("Testing shape:", temporal_test_with_topics.shape)
print("New topic features:", len(all_topics))
print(
    "Training topic features with nonzero values:",
    (temporal_train_with_topics[
        [c for c in temporal_train_with_topics.columns if c.startswith("topic_")]
    ].sum() > 0).sum()
)


Training shape: (123018, 48)
Testing shape: (139737, 48)
New topic features: 38
Training topic features with nonzero values: 37


In [70]:

print("Training columns:")
print(temporal_train_with_topics.columns.tolist())

print("\nLabel counts in training:")
print(temporal_train_with_topics["label"].value_counts(dropna=False))

print("\nLabel counts in testing:")
print(temporal_test_with_topics["label"].value_counts(dropna=False))

print("\nMissing training labels:", temporal_train_with_topics["label"].isna().sum())
print("Missing testing labels:", temporal_test_with_topics["label"].isna().sum())


Training columns:
['user', 'problem', 'label', 'problem_rating', 'rating_gap', 'absolute_rating_gap', 'historical_solved_count', 'problem_tag_count', 'familiar_tag_overlap', 'familiar_tag_ratio', 'topic_special_special', 'topic_2_sat', 'topic_binary_search', 'topic_bitmasks', 'topic_brute_force', 'topic_chinese_remainder_theorem', 'topic_combinatorics', 'topic_communication', 'topic_constructive_algorithms', 'topic_data_structures', 'topic_dfs_and_similar', 'topic_divide_and_conquer', 'topic_dp', 'topic_dsu', 'topic_expression_parsing', 'topic_fft', 'topic_flows', 'topic_games', 'topic_geometry', 'topic_graph_matchings', 'topic_graphs', 'topic_greedy', 'topic_hashing', 'topic_implementation', 'topic_interactive', 'topic_math', 'topic_matrices', 'topic_meet_in_the_middle', 'topic_number_theory', 'topic_probabilities', 'topic_schedules', 'topic_shortest_paths', 'topic_sortings', 'topic_string_suffix_structures', 'topic_strings', 'topic_ternary_search', 'topic_trees', 'topic_two_pointers'

In [71]:

# Columns that describe the problem and user's topic familiarity.
base_features = [
    "problem_rating",
    "rating_gap",
    "absolute_rating_gap",
    "historical_solved_count",
    "problem_tag_count",
    "familiar_tag_overlap",
    "familiar_tag_ratio",
]

topic_features = [
    column
    for column in temporal_train_with_topics.columns
    if column.startswith("topic_")
]

feature_columns = base_features + topic_features

# Align train and test to exactly the same feature columns.
X_train_topics = temporal_train_with_topics[feature_columns].copy()
X_test_topics = temporal_test_with_topics[feature_columns].copy()

y_train_topics = temporal_train_with_topics["label"].astype(int)
y_test_topics = temporal_test_with_topics["label"].astype(int)

print("Training matrix:", X_train_topics.shape)
print("Testing matrix:", X_test_topics.shape)
print("Feature count:", len(feature_columns))
print("Same feature order:", list(X_train_topics.columns) == list(X_test_topics.columns))
print("Missing training values:", X_train_topics.isna().sum().sum())
print("Missing testing values:", X_test_topics.isna().sum().sum())


Training matrix: (123018, 45)
Testing matrix: (139737, 45)
Feature count: 45
Same feature order: True
Missing training values: 0
Missing testing values: 0


In [72]:

import lightgbm as lgb
import numpy as np

# Group sizes: number of candidate problems for each user.
train_group_topics = (
    temporal_train_with_topics.groupby("user", sort=False)
    .size()
    .to_numpy()
)

test_group_topics = (
    temporal_test_with_topics.groupby("user", sort=False)
    .size()
    .to_numpy()
)

# LambdaRank requires candidates grouped by user.
train_order = temporal_train_with_topics.sort_values(
    "user", kind="stable"
).index
test_order = temporal_test_with_topics.sort_values(
    "user", kind="stable"
).index

train_rank_df = temporal_train_with_topics.loc[train_order].reset_index(drop=True)
test_rank_df = temporal_test_with_topics.loc[test_order].reset_index(drop=True)

X_train_rank = train_rank_df[feature_columns]
y_train_rank = train_rank_df["label"].astype(int)

X_test_rank = test_rank_df[feature_columns]
y_test_rank = test_rank_df["label"].astype(int)

train_group_topics = train_rank_df.groupby("user", sort=False).size().to_list()
test_group_topics = test_rank_df.groupby("user", sort=False).size().to_list()

topic_ranker = lgb.LGBMRanker(
    objective="lambdarank",
    metric="ndcg",
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    verbosity=-1
)

topic_ranker.fit(
    X_train_rank,
    y_train_rank,
    group=train_group_topics,
    eval_set=[(X_test_rank, y_test_rank)],
    eval_group=[test_group_topics],
    eval_at=[10]
)

print("Topic-aware ranker trained.")
print("Training users:", len(train_group_topics))
print("Testing users:", len(test_group_topics))
print("Features used:", len(feature_columns))


C:\Users\GARIMA YADUVANSHI\OneDrive\Desktop\DSAForge\DSAForge\backend\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Topic-aware ranker trained.
Training users: 270
Testing users: 295
Features used: 45


In [73]:

print("Topic-aware model trained:", topic_ranker is not None)
print("Training feature count:", topic_ranker.n_features_in_)

topic_predictions = topic_ranker.predict(X_test_rank)

print("Predictions generated:", len(topic_predictions))
print("All predictions finite:", np.isfinite(topic_predictions).all())
print("Unique prediction scores:", len(np.unique(topic_predictions)))


Topic-aware model trained: True
Training feature count: 45
Predictions generated: 139737
All predictions finite: True
Unique prediction scores: 96178


In [74]:

import numpy as np

# Baseline model: original 7 features, same test candidates
baseline_predictions = temporal_model.predict(
    X_test_rank[base_features]
)

def ranking_metrics(df, scores, k=10):
    ndcg_values = []
    recall_values = []
    offset = 0

    for _, group in df.groupby("user", sort=False):
        n = len(group)
        labels = group["label"].to_numpy()
        user_scores = scores[offset:offset + n]
        offset += n

        top_indices = np.argsort(-user_scores, kind="stable")[:k]
        top_labels = labels[top_indices]

        discounts = np.log2(np.arange(2, len(top_labels) + 2))
        dcg = np.sum(top_labels / discounts)

        ideal_labels = np.sort(labels)[::-1][:k]
        ideal_discounts = np.log2(
            np.arange(2, len(ideal_labels) + 2)
        )
        idcg = np.sum(ideal_labels / ideal_discounts)

        if idcg > 0:
            ndcg_values.append(dcg / idcg)

        total_relevant = labels.sum()
        if total_relevant > 0:
            recall_values.append(
                top_labels.sum() / total_relevant
            )

    return {
        f"NDCG@{k}": np.mean(ndcg_values),
        f"Recall@{k}": np.mean(recall_values)
    }

baseline_metrics = ranking_metrics(
    test_rank_df, baseline_predictions, k=10
)

topic_metrics = ranking_metrics(
    test_rank_df, topic_predictions, k=10
)

print("BASELINE (7 features):", baseline_metrics)
print("TOPIC-AWARE (45 features):", topic_metrics)

print("\nImprovement:")
for metric in baseline_metrics:
    difference = topic_metrics[metric] - baseline_metrics[metric]
    print(f"{metric}: {difference:+.5f}")


BASELINE (7 features): {'NDCG@10': np.float64(0.6453569483023636), 'Recall@10': np.float64(0.11754149976261355)}
TOPIC-AWARE (45 features): {'NDCG@10': np.float64(0.45090085951547365), 'Recall@10': np.float64(0.10486920652299253)}

Improvement:
NDCG@10: -0.19446
Recall@10: -0.01267


In [75]:

baseline_ranker_fair = lgb.LGBMRanker(
    objective="lambdarank",
    metric="ndcg",
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    verbosity=-1
)

baseline_ranker_fair.fit(
    train_rank_df[base_features],
    y_train_rank,
    group=train_group_topics
)

fair_baseline_predictions = baseline_ranker_fair.predict(
    test_rank_df[base_features]
)

fair_baseline_metrics = ranking_metrics(
    test_rank_df, fair_baseline_predictions, k=10
)

fair_topic_metrics = ranking_metrics(
    test_rank_df, topic_predictions, k=10
)

print("FAIR BASELINE (7 features):", fair_baseline_metrics)
print("TOPIC-AWARE (45 features):", fair_topic_metrics)

print("\nTopic-aware minus fair baseline:")
for metric in fair_baseline_metrics:
    difference = fair_topic_metrics[metric] - fair_baseline_metrics[metric]
    print(f"{metric}: {difference:+.5f}")


FAIR BASELINE (7 features): {'NDCG@10': np.float64(0.6357746971757197), 'Recall@10': np.float64(0.12181473747376662)}
TOPIC-AWARE (45 features): {'NDCG@10': np.float64(0.45090085951547365), 'Recall@10': np.float64(0.10486920652299253)}

Topic-aware minus fair baseline:
NDCG@10: -0.18487
Recall@10: -0.01695


In [76]:

# Check whether problems accepted in Jul-Dec 2023
# were incorrectly used as negative training examples.

future_accepted = (
    test_labels_df[["user", "problem"]]
    .drop_duplicates()
    .assign(future_accepted=True)
)

training_check = temporal_train_df[
    ["user", "problem", "label"]
].merge(
    future_accepted,
    on=["user", "problem"],
    how="inner"
)

false_negative_check = training_check[
    (training_check["label"] == 0)
    & (training_check["future_accepted"])
]

print("Future accepted user-problem pairs:", len(future_accepted))
print("Overlapping training rows:", len(training_check))
print("Potential false-negative training rows:", len(false_negative_check))


Future accepted user-problem pairs: 33941
Overlapping training rows: 1704
Potential false-negative training rows: 1704


In [77]:

# 1. Identify future-accepted pairs so they are not treated as negatives.
future_pairs = set(
    zip(test_labels_df["user"], test_labels_df["problem"])
)

is_false_negative = (
    temporal_train_with_topics["label"].eq(0)
    & pd.Series(
        [
            (u, p) in future_pairs
            for u, p in zip(
                temporal_train_with_topics["user"],
                temporal_train_with_topics["problem"]
            )
        ],
        index=temporal_train_with_topics.index
    )
)

clean_train_df = (
    temporal_train_with_topics.loc[~is_false_negative]
    .sort_values("user", kind="stable")
    .reset_index(drop=True)
)

# 2. Prepare cleaned training labels and user-wise groups.
X_clean_base = clean_train_df[base_features]
X_clean_topics = clean_train_df[feature_columns]
y_clean = clean_train_df["label"].astype(int)

clean_groups = clean_train_df.groupby("user", sort=False).size().to_list()

print("Rows removed:", int(is_false_negative.sum()))
print("Clean training rows:", len(clean_train_df))
print("Remaining training labels:")
print(y_clean.value_counts())

# 3. Train both models on exactly the same cleaned rows.
baseline_clean = lgb.LGBMRanker(
    objective="lambdarank",
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    verbosity=-1
)

topic_clean = lgb.LGBMRanker(
    objective="lambdarank",
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    verbosity=-1
)

baseline_clean.fit(
    X_clean_base, y_clean, group=clean_groups
)

topic_clean.fit(
    X_clean_topics, y_clean, group=clean_groups
)

# 4. Evaluate on the same held-out test rows.
baseline_clean_scores = baseline_clean.predict(
    test_rank_df[base_features]
)

topic_clean_scores = topic_clean.predict(
    test_rank_df[feature_columns]
)

baseline_clean_metrics = ranking_metrics(
    test_rank_df, baseline_clean_scores, k=10
)

topic_clean_metrics = ranking_metrics(
    test_rank_df, topic_clean_scores, k=10
)

print("\nCLEAN BASELINE:", baseline_clean_metrics)
print("CLEAN TOPIC MODEL:", topic_clean_metrics)


Rows removed: 1704
Clean training rows: 121314
Remaining training labels:
label
0    91864
1    29450
Name: count, dtype: int64

CLEAN BASELINE: {'NDCG@10': np.float64(0.6312312965037011), 'Recall@10': np.float64(0.12059000312621902)}
CLEAN TOPIC MODEL: {'NDCG@10': np.float64(0.47433103584701275), 'Recall@10': np.float64(0.10690601341370136)}


In [78]:

import inspect

print(inspect.getsource(build_examples))



def build_examples(users, positive_map, historical_map, tag_map, catalogue, rng):
    examples = []

    for user in users:
        if user not in historical_map or not positive_map[user]:
            continue

        past_ratings = [
            float(row.problem_rating)
            for row in historical_map[user]
            if pd.notna(row.problem_rating)
        ]
        if not past_ratings:
            continue

        median_rating = float(np.median(past_ratings))
        familiar = tag_map[user]
        past_problems = {row.problem for row in historical_map[user]}
        positives = positive_map[user]

        blocked = past_problems | positives
        negative_pool = [p for p in catalogue if p not in blocked]
        n_negatives = min(len(negative_pool), max(100, 3 * len(positives)))
        negatives = rng.sample(negative_pool, n_negatives)

        for problem_id, label in (
            [(p, 1) for p in positives] +
            [(p, 0) for p in negatives]
        ):
    

In [79]:

import random
import pandas as pd
import numpy as np

# Combine accepted problems across both labelled periods.
all_positive_map = {}

for source in (train_positives, test_positives):
    for user, problems in source.items():
        all_positive_map.setdefault(user, set()).update(problems)

def build_examples_corrected(
    users, positive_map, historical_map,
    tag_map, catalogue, known_positive_map, rng
):
    examples = []

    for user in users:
        positives = set(positive_map.get(user, []))
        if user not in historical_map or not positives:
            continue

        past_ratings = [
            float(row.problem_rating)
            for row in historical_map[user]
            if pd.notna(row.problem_rating)
        ]
        if not past_ratings:
            continue

        median_rating = float(np.median(past_ratings))
        familiar = tag_map.get(user, {})
        past_problems = {
            row.problem for row in historical_map[user]
        }

        # Exclude all known accepted problems from negative sampling.
        blocked = (
            past_problems
            | set(known_positive_map.get(user, set()))
        )

        negative_pool = [
            p for p in catalogue if p not in blocked
        ]
        n_negatives = min(
            len(negative_pool),
            max(100, 3 * len(positives))
        )
        negatives = rng.sample(negative_pool, n_negatives)

        for problem_id, label in (
            [(p, 1) for p in positives]
            + [(p, 0) for p in negatives]
        ):
            problem = catalogue[problem_id]
            tags = problem["tags"]
            overlap = sum(
                1 for tag in tags if familiar.get(tag, 0) > 0
            )

            examples.append({
                "user": user,
                "problem": problem_id,
                "label": label,
                "problem_rating": problem["rating"],
                "rating_gap": problem["rating"] - median_rating,
                "absolute_rating_gap": abs(
                    problem["rating"] - median_rating
                ),
                "historical_solved_count": len(
                    historical_map[user]
                ),
                "problem_tag_count": len(tags),
                "familiar_tag_overlap": overlap,
                "familiar_tag_ratio": overlap / len(tags)
            })

    return pd.DataFrame(examples)

print("Corrected function defined.")
print("Users with known accepted problems:", len(all_positive_map))


Corrected function defined.
Users with known accepted problems: 299


In [80]:

import pandas as pd
from collections import defaultdict

TRAIN_CUTOFF = 1672531200  # 2023-01-01
TEST_CUTOFF = 1688169600   # 2023-07-01

print("History columns:")
print(history_df.columns.tolist())
print("\nSample history row:")
display(history_df.head(1))


History columns:
['user', 'problem', 'problem_rating', 'tags']

Sample history row:


,user,problem,problem_rating,tags
0,user3106,102B,1000.0,"[""implementation""]"


In [81]:

tables = pd.read_sql_query("""
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name
""", conn)

display(tables)

for table_name in tables["name"]:
    if any(word in table_name.lower() for word in ["accepted", "submission", "time", "history"]):
        safe_name = table_name.replace('"', '""')
        print(f"\nTable: {table_name}")
        print(pd.read_sql_query(
            f'PRAGMA table_info("{safe_name}")',
            conn
        )[["name", "type"]].to_string(index=False))


,name
0,accepted_problem_times
1,codeforces_api_problems
2,problem_catalog
3,user_problem_features



Table: accepted_problem_times
       name    type
       user    TEXT
    problem    TEXT
accepted_at INTEGER


In [82]:

# Unix timestamps: seconds
TRAIN_CUTOFF = 1672531200  # 2023-01-01
TEST_CUTOFF = 1688169600   # 2023-07-01

history_users = history_df["user"].drop_duplicates().tolist()
placeholders = ",".join("?" for _ in history_users)

query = f"""
SELECT
    a.user,
    a.problem,
    p.problem_rating,
    p.tags,
    a.accepted_at
FROM accepted_problem_times AS a
LEFT JOIN codeforces_api_problems AS p
    ON a.problem = p.problem_id
WHERE a.user IN ({placeholders})
  AND a.accepted_at < ?
ORDER BY a.user, a.accepted_at
"""

history_train_df = pd.read_sql_query(
    query, conn, params=history_users + [TRAIN_CUTOFF]
)

history_test_df = pd.read_sql_query(
    query, conn, params=history_users + [TEST_CUTOFF]
)

print("Pre-2023 training history:", history_train_df.shape)
print("Pre-July testing history:", history_test_df.shape)

print(
    "Training history latest timestamp:",
    history_train_df["accepted_at"].max()
)
print(
    "Testing history latest timestamp:",
    history_test_df["accepted_at"].max()
)

display(history_test_df.head(3))


Pre-2023 training history: (109798, 5)
Pre-July testing history: (143033, 5)
Training history latest timestamp: 1672530138
Testing history latest timestamp: 1688168582


,user,problem,problem_rating,tags,accepted_at
0,user10000,1624A,800.0,"[""math""]",1643120749
1,user10000,1669A,800.0,"[""implementation""]",1650706305
2,user10000,1674A,800.0,"[""constructive algorithms"", ""math""]",1651581434


In [83]:

import json
import random
from collections import Counter, defaultdict

def make_history_map(df):
    result = defaultdict(list)
    for row in df.itertuples(index=False):
        result[row.user].append(row)
    return result

def make_tag_map(history_map):
    result = defaultdict(Counter)

    for user, rows in history_map.items():
        for row in rows:
            try:
                tags = json.loads(row.tags) if isinstance(row.tags, str) else []
            except (json.JSONDecodeError, TypeError):
                tags = []

            if isinstance(tags, list):
                result[user].update(tags)

    return result

# Use only information available by each respective cutoff.
history_train_map = make_history_map(history_train_df)
history_test_map = make_history_map(history_test_df)

tag_train_map = make_tag_map(history_train_map)
tag_test_map = make_tag_map(history_test_map)

# Known accepted pairs are excluded from negative sampling so they
# are not deliberately labelled negative in this retrospective dataset.
# They are NOT used as model features.
all_known_positives = {}
for source in (train_positives, test_positives):
    for user, problems in source.items():
        all_known_positives.setdefault(user, set()).update(problems)

# Rebuild with independent random generators for reproducibility.
rebuilt_train_df = build_examples_corrected(
    users=list(train_positives.keys()),
    positive_map=train_positives,
    historical_map=history_train_map,
    tag_map=tag_train_map,
    catalogue=catalogue,
    known_positive_map=all_known_positives,
    rng=random.Random(42)
)

rebuilt_test_df = build_examples_corrected(
    users=list(test_positives.keys()),
    positive_map=test_positives,
    historical_map=history_test_map,
    tag_map=tag_test_map,
    catalogue=catalogue,
    known_positive_map=all_known_positives,
    rng=random.Random(43)
)

print("Rebuilt training:", rebuilt_train_df.shape)
print("Rebuilt testing:", rebuilt_test_df.shape)
print("\nTraining labels:")
print(rebuilt_train_df["label"].value_counts())
print("\nTesting labels:")
print(rebuilt_test_df["label"].value_counts())


Rebuilt training: (123018, 10)
Rebuilt testing: (140377, 10)

Training labels:
label
0    93568
1    29450
Name: count, dtype: int64

Testing labels:
label
0    106476
1     33901
Name: count, dtype: int64


In [84]:

future_pairs = set(
    zip(test_labels_df["user"], test_labels_df["problem"])
)

remaining_false_negatives = rebuilt_train_df[
    (rebuilt_train_df["label"] == 0)
    & rebuilt_train_df.apply(
        lambda row: (row["user"], row["problem"]) in future_pairs,
        axis=1
    )
]

print("Remaining false-negative rows:", len(remaining_false_negatives))

assert len(remaining_false_negatives) == 0, (
    "False negatives remain. Do not train yet."
)

print("PASS: No training negatives overlap with known test positives.")


Remaining false-negative rows: 0
PASS: No training negatives overlap with known test positives.


In [85]:
import numpy as np
import lightgbm as lgb

# Use the same seven baseline features for both datasets.
feature_columns = [
    "problem_rating",
    "rating_gap",
    "absolute_rating_gap",
    "historical_solved_count",
    "problem_tag_count",
    "familiar_tag_overlap",
    "familiar_tag_ratio",
]

# Keep each user's candidate problems together for ranking.
train_data = rebuilt_train_df.sort_values(
    "user", kind="stable"
).reset_index(drop=True)

test_data = rebuilt_test_df.sort_values(
    "user", kind="stable"
).reset_index(drop=True)

X_train = train_data[feature_columns]
y_train = train_data["label"].astype(int)

X_test = test_data[feature_columns]
y_test = test_data["label"].astype(int)

train_groups = train_data.groupby("user", sort=False).size().tolist()

# Train the baseline ranker.
baseline_corrected = lgb.LGBMRanker(
    objective="lambdarank",
    metric="ndcg",
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    verbosity=-1,
)

baseline_corrected.fit(
    X_train,
    y_train,
    group=train_groups,
)

# Evaluate using the existing ranking_metrics function.
test_scores = baseline_corrected.predict(X_test)

results = ranking_metrics(test_data, test_scores, k=10)

print("Training rows:", len(train_data))
print("Testing rows:", len(test_data))
print("Training users:", train_data["user"].nunique())
print("Testing users:", test_data["user"].nunique())
print("Features:", len(feature_columns))
print("Corrected baseline results:", results)

Training rows: 123018
Testing rows: 140377
Training users: 270
Testing users: 298
Features: 7
Corrected baseline results: {'NDCG@10': np.float64(0.6251099766697693), 'Recall@10': np.float64(0.1242293460173196)}


In [86]:
# Check whether test candidates include problems
# already solved before the test period began.

previously_solved_pairs = {
    (user, row.problem)
    for user, rows in history_test_map.items()
    for row in rows
}

test_candidate_pairs = set(
    zip(test_data["user"], test_data["problem"])
)

overlap = test_candidate_pairs & previously_solved_pairs

print("Test candidate pairs:", len(test_candidate_pairs))
print("Previously solved pairs included:", len(overlap))

if overlap:
    print("Sample overlaps:", list(overlap)[:10])
else:
    print("PASS: No previously solved problems in test candidates.")

Test candidate pairs: 140377
Previously solved pairs included: 0
PASS: No previously solved problems in test candidates.


In [88]:
problem_details = pd.read_sql_query(
    """
    SELECT
        problem_id,
        problem_name,
        problem_rating,
        tags
    FROM codeforces_api_problems
    """,
    conn
)

print("Problem details loaded:", len(problem_details))
display(problem_details.head())

Problem details loaded: 11450


,problem_id,problem_name,problem_rating,tags
0,2275H,A Problem to Warm Up the Eyebrows,NaN,"[""combinatorics"", ""dp"", ""math""]"
1,2275G,Copper Squander,NaN,"[""binary search"", ""data structures"", ""dsu"", ""g..."
2,2275F,Tea Blend,NaN,"[""hashing"", ""math"", ""number theory""]"
3,2275E,Repentance Is Already on the Way,NaN,"[""dp"", ""implementation""]"
4,2275D,Precision Alignment,NaN,"[""binary search"", ""brute force"", ""greedy"", ""im..."


In [89]:
# Select one test user and rank their candidate problems.
user_id = test_data["user"].iloc[0]

user_rows = test_data[test_data["user"] == user_id].copy()
user_rows["predicted_score"] = baseline_corrected.predict(
    user_rows[feature_columns]
)

top10 = user_rows.sort_values(
    "predicted_score", ascending=False
).head(10)

# Attach problem names, ratings and tags.
details = problem_details.drop_duplicates("problem_id")

top10 = top10.merge(
    details[["problem_id", "problem_name", "problem_rating", "tags"]],
    left_on="problem",
    right_on="problem_id",
    how="left",
    suffixes=("", "_catalogue")
)

display(
    top10[
        [
            "user",
            "problem",
            "problem_name",
            "problem_rating",
            "tags",
            "predicted_score",
            "label"
        ]
    ].reset_index(drop=True)
)


,user,problem,problem_name,problem_rating,tags,predicted_score,label
0,user10000,1324C,Frog Jumps,1100.0,"[""binary search"", ""data structures"", ""dfs and ...",1.135147,0
1,user10000,1917B,Erase First or Second Letter,1100.0,"[""brute force"", ""combinatorics"", ""data structu...",1.135147,1
2,user10000,1904B,Collecting Game,1100.0,"[""binary search"", ""dp"", ""greedy"", ""sortings"", ...",1.135147,1
3,user10000,1976A,Verify Password,800.0,"[""implementation"", ""sortings"", ""strings""]",0.797600,0
4,user10000,1884A,Simple Design,800.0,"[""brute force"", ""greedy"", ""math""]",0.797600,1
5,user10000,1915B,Not Quite Latin Square,800.0,"[""bitmasks"", ""brute force"", ""implementation""]",0.797600,1
6,user10000,1152A,Neko Finds Grapes,800.0,"[""greedy"", ""implementation"", ""math""]",0.797600,0
7,user10000,1898A,Milica and String,800.0,"[""brute force"", ""implementation"", ""strings""]",0.797600,1
8,user10000,1491A,K-th Largest Value,800.0,"[""brute force"", ""greedy"", ""implementation""]",0.797600,0
9,user10000,1899A,Game with Integers,800.0,"[""games"", ""math"", ""number theory""]",0.797600,1


In [90]:
# Diagnose the current baseline ranker.

importance = pd.DataFrame({
    "feature": feature_columns,
    "importance": baseline_corrected.feature_importances_
}).sort_values("importance", ascending=False)

test_scores_all = baseline_corrected.predict(X_test)

print("Feature importance:")
display(importance.reset_index(drop=True))

print("\nUnique scores across all test candidates:",
      len(np.unique(test_scores_all)), "out of", len(test_scores_all))

print("\nTop 10 score distribution for this user:")
display(
    top10[["problem", "problem_rating", "predicted_score", "label"]]
    .groupby("predicted_score")
    .agg(
        problems=("problem", "count"),
        ratings=("problem_rating", "unique"),
        positive_labels=("label", "sum")
    )
    .reset_index()
    .sort_values("predicted_score", ascending=False)
)

Feature importance:


,feature,importance
0,historical_solved_count,3321
1,problem_rating,1682
2,rating_gap,1529
3,problem_tag_count,885
4,familiar_tag_overlap,579
5,familiar_tag_ratio,572
6,absolute_rating_gap,432



Unique scores across all test candidates: 32515 out of 140377

Top 10 score distribution for this user:


,predicted_score,problems,ratings,positive_labels
1,1.135147,3,[1100.0],2
0,0.797600,7,[800.0],4


In [91]:
# Compare LightGBM against a rating-only ranking baseline.

rating_scores = -np.abs(test_data["rating_gap"].to_numpy())

lightgbm_scores = baseline_corrected.predict(
    test_data[feature_columns]
)

rating_results = ranking_metrics(
    test_data, rating_scores, k=10
)

lightgbm_results = ranking_metrics(
    test_data, lightgbm_scores, k=10
)

comparison = pd.DataFrame(
    [rating_results, lightgbm_results],
    index=["Rating-only baseline", "LightGBM"]
)

print("Same test candidates:", len(test_data))
print("Same test users:", test_data["user"].nunique())
display(comparison)

Same test candidates: 140377
Same test users: 298


,NDCG@10,Recall@10
Rating-only baseline,0.701341,0.125359
LightGBM,0.625110,0.124229


In [92]:
import json
import numpy as np
import pandas as pd

# Build tag-preference scores using each user's pre-test history.
def get_tag_preference_score(user, problem_id):
    user_tags = tag_test_map.get(user, {})
    problem_tags = catalogue.get(problem_id, {}).get("tags", [])

    if not problem_tags:
        return 0.0

    total = sum(user_tags.values())
    if total == 0:
        return 0.0

    # Normalize tag counts so users with longer histories
    # do not automatically receive larger scores.
    return sum(user_tags.get(tag, 0) / total for tag in problem_tags) / len(problem_tags)


tag_test_scores = np.array([
    get_tag_preference_score(user, problem)
    for user, problem in zip(test_data["user"], test_data["problem"])
])

tag_results = ranking_metrics(test_data, tag_test_scores, k=10)

comparison = pd.DataFrame(
    [
        rating_results,
        lightgbm_results,
        tag_results,
    ],
    index=[
        "Rating-only baseline",
        "LightGBM",
        "Tag-preference baseline",
    ],
)

display(comparison)

,NDCG@10,Recall@10
Rating-only baseline,0.701341,0.125359
LightGBM,0.625110,0.124229
Tag-preference baseline,0.410360,0.062633


In [93]:
# Combine rating suitability with a small tag-preference signal.
# Both score arrays must align with test_data row order.

rating_scores = -np.abs(test_data["rating_gap"].to_numpy())

tag_scores = tag_test_scores.copy()

# Normalize tag scores to [0, 1] to keep the contribution controlled.
max_tag_score = tag_scores.max()
if max_tag_score > 0:
    tag_scores = tag_scores / max_tag_score

hybrid_scores = rating_scores + 0.2 * tag_scores

hybrid_results = ranking_metrics(
    test_data, hybrid_scores, k=10
)

comparison = pd.DataFrame(
    [
        rating_results,
        tag_results,
        hybrid_results,
    ],
    index=[
        "Rating-only baseline",
        "Tag-preference baseline",
        "Hybrid: rating + tag",
    ],
)

display(comparison)

,NDCG@10,Recall@10
Rating-only baseline,0.701341,0.125359
Tag-preference baseline,0.410360,0.062633
Hybrid: rating + tag,0.348224,0.086301


In [94]:
# Generate top 10 rating-matched, previously unsolved candidates.
user_id = test_data["user"].iloc[0]

history_solved = {
    row.problem for row in history_test_map.get(user_id, [])
}

user_candidates = test_data[
    (test_data["user"] == user_id)
    & (~test_data["problem"].isin(history_solved))
].copy()

user_candidates["rating_only_score"] = -np.abs(
    user_candidates["rating_gap"]
)

recommended = user_candidates.sort_values(
    "rating_only_score", ascending=False
).head(10)

recommended = recommended.merge(
    problem_details[
        ["problem_id", "problem_name", "problem_rating", "tags"]
    ].drop_duplicates("problem_id"),
    left_on="problem",
    right_on="problem_id",
    how="left"
)

display(
    recommended[
        ["problem", "problem_name", "problem_rating", "tags"]
    ].reset_index(drop=True)
)

KeyError: "['problem_rating'] not in index"

In [95]:
display(
    recommended[
        ["problem", "problem_name", "tags"]
    ].assign(
        rating=recommended["problem"].map(
            problem_details.drop_duplicates("problem_id")
            .set_index("problem_id")["problem_rating"]
        )
    ).reset_index(drop=True)
)

,problem,problem_name,tags,rating
0,1915C,Can I Square?,"[""binary search"", ""implementation""]",800.0
1,472A,Design Tutorial: Learn from Math,"[""math"", ""number theory""]",800.0
2,1857A,Array Coloring,"[""greedy"", ""math""]",800.0
3,1846A,Rudolph and Cut the Rope,"[""implementation"", ""math""]",800.0
4,1902A,Binary Imbalance,"[""constructive algorithms""]",800.0
5,1907A,Rook,"[""implementation""]",800.0
6,1898A,Milica and String,"[""brute force"", ""implementation"", ""strings""]",800.0
7,1779A,Hall of Fame,"[""constructive algorithms"", ""greedy"", ""strings""]",800.0
8,1873A,Short Sort,"[""brute force"", ""implementation""]",800.0
9,1849A,Morning Sandwich,"[""implementation"", ""math""]",800.0


In [96]:
# Record the current benchmark for future model comparisons.

baseline_benchmark = pd.DataFrame([
    {
        "model": "Rating-only baseline",
        "NDCG@10": float(rating_results["NDCG@10"]),
        "Recall@10": float(rating_results["Recall@10"]),
    },
    {
        "model": "LightGBM (7 features)",
        "NDCG@10": float(lightgbm_results["NDCG@10"]),
        "Recall@10": float(lightgbm_results["Recall@10"]),
    },
    {
        "model": "Tag-preference baseline",
        "NDCG@10": float(tag_results["NDCG@10"]),
        "Recall@10": float(tag_results["Recall@10"]),
    },
    {
        "model": "Hybrid (rating + tags)",
        "NDCG@10": float(hybrid_results["NDCG@10"]),
        "Recall@10": float(hybrid_results["Recall@10"]),
    },
])

display(
    baseline_benchmark.sort_values(
        "NDCG@10", ascending=False
    ).reset_index(drop=True)
)

,model,NDCG@10,Recall@10
0,Rating-only baseline,0.701341,0.125359
1,LightGBM (7 features),0.625110,0.124229
2,Tag-preference baseline,0.410360,0.062633
3,Hybrid (rating + tags),0.348224,0.086301


In [97]:
def recommend_problems_rating_only(
    user_id,
    history_map,
    catalogue,
    top_n=10
):
    # Get this user's historical solved problems.
    history = history_map.get(user_id, [])
    if not history:
        return pd.DataFrame(
            columns=["problem", "rating", "rating_gap", "tags"]
        )

    past_ratings = [
        float(row.problem_rating)
        for row in history
        if pd.notna(row.problem_rating)
    ]
    if not past_ratings:
        return pd.DataFrame(
            columns=["problem", "rating", "rating_gap", "tags"]
        )

    median_rating = float(np.median(past_ratings))
    solved = {row.problem for row in history}

    candidates = []
    for problem_id, details in catalogue.items():
        if problem_id in solved:
            continue

        rating = details.get("rating")
        if rating is None or pd.isna(rating):
            continue

        candidates.append({
            "problem": problem_id,
            "rating": float(rating),
            "rating_gap": float(rating) - median_rating,
            "tags": ", ".join(details.get("tags", [])),
        })

    result = pd.DataFrame(candidates)
    if result.empty:
        return result

    result["absolute_rating_gap"] = result["rating_gap"].abs()

    return (
        result.sort_values(
            ["absolute_rating_gap", "problem"],
            ascending=[True, True]
        )
        .head(top_n)
        .drop(columns="absolute_rating_gap")
        .reset_index(drop=True)
    )


# Quick test on one user with pre-test history.
sample_user = test_data["user"].iloc[0]

sample_recommendations = recommend_problems_rating_only(
    user_id=sample_user,
    history_map=history_test_map,
    catalogue=catalogue,
    top_n=10
)

print("Sample user:", sample_user)
display(sample_recommendations)

Sample user: user10000


,problem,rating,rating_gap,tags
0,1002A1,800.0,0.0,*special
1,1003A,800.0,0.0,implementation
2,1005A,800.0,0.0,implementation
3,1006A,800.0,0.0,implementation
4,1009A,800.0,0.0,implementation
5,1013A,800.0,0.0,math
6,1015A,800.0,0.0,implementation
7,1017A,800.0,0.0,implementation
8,1028A,800.0,0.0,implementation
9,1031A,800.0,0.0,"implementation, math"


In [98]:
# Validate recommendation generation across 20 test users.

sample_users = test_data["user"].drop_duplicates().head(20).tolist()

validation_rows = []

for user_id in sample_users:
    recs = recommend_problems_rating_only(
        user_id=user_id,
        history_map=history_test_map,
        catalogue=catalogue,
        top_n=10
    )

    solved_ids = {
        row.problem for row in history_test_map.get(user_id, [])
    }

    validation_rows.append({
        "user": user_id,
        "recommendations": len(recs),
        "duplicate_problems": int(recs["problem"].duplicated().sum()),
        "already_solved": int(recs["problem"].isin(solved_ids).sum()),
        "median_recommended_rating": (
            float(recs["rating"].median()) if not recs.empty else np.nan
        )
    })

validation_df = pd.DataFrame(validation_rows)
display(validation_df)

print("\nTotal recommendations:", validation_df["recommendations"].sum())
print("Already-solved recommendations:", validation_df["already_solved"].sum())
print("Users receiving 10 recommendations:",
      int((validation_df["recommendations"] == 10).sum()))

,user,recommendations,duplicate_problems,already_solved,median_recommended_rating
0,user10000,10,0,0,800.0
1,user10046,10,0,0,800.0
2,user10087,10,0,0,800.0
3,user101,10,0,0,1100.0
4,user10149,10,0,0,1700.0
5,user10169,10,0,0,1300.0
6,user10192,10,0,0,1300.0
7,user10237,10,0,0,1200.0
8,user10248,10,0,0,800.0
9,user1026,10,0,0,800.0



Total recommendations: 200
Already-solved recommendations: 0
Users receiving 10 recommendations: 20


In [99]:
# Attach names and build direct Codeforces problem links.

details = problem_details.drop_duplicates("problem_id").copy()

# Use problem_id as the join key.
recommended_with_details = sample_recommendations.merge(
    details[["problem_id", "problem_name", "problem_rating", "tags"]],
    left_on="problem",
    right_on="problem_id",
    how="left",
    suffixes=("", "_catalogue")
)

recommended_with_details["problem_url"] = (
    "https://codeforces.com/problemset/problem/"
    + recommended_with_details["problem"].str.extract(r"(\d+)")[0]
    + "/"
    + recommended_with_details["problem"].str.extract(r"\d+([A-Z]\d*)$")[0]
)

display(
    recommended_with_details[
        ["problem", "problem_name", "rating", "tags", "problem_url"]
    ].reset_index(drop=True)
)

,problem,problem_name,rating,tags,problem_url
0,1002A1,Generate superposition of all basis states,800.0,*special,https://codeforces.com/problemset/problem/1002/A1
1,1003A,Polycarp's Pockets,800.0,implementation,https://codeforces.com/problemset/problem/1003/A
2,1005A,Tanya and Stairways,800.0,implementation,https://codeforces.com/problemset/problem/1005/A
3,1006A,Adjacent Replacements,800.0,implementation,https://codeforces.com/problemset/problem/1006/A
4,1009A,Game Shopping,800.0,implementation,https://codeforces.com/problemset/problem/1009/A
5,1013A,Piles With Stones,800.0,math,https://codeforces.com/problemset/problem/1013/A
6,1015A,Points in Segments,800.0,implementation,https://codeforces.com/problemset/problem/1015/A
7,1017A,The Rank,800.0,implementation,https://codeforces.com/problemset/problem/1017/A
8,1028A,Find Square,800.0,implementation,https://codeforces.com/problemset/problem/1028/A
9,1031A,Golden Plate,800.0,"implementation, math",https://codeforces.com/problemset/problem/1031/A


In [100]:

# Build personalized user-history features

from collections import Counter
import numpy as np
import pandas as pd
import json

def build_user_profile(history):
    # Sort accepted problems from oldest to newest
    rows = sorted(history, key=lambda row: row.accepted_at)

    ratings = [
        float(row.problem_rating)
        for row in rows
        if pd.notna(row.problem_rating)
    ]

    recent_ratings = ratings[-20:]

    if ratings:
        median_rating = float(np.median(ratings))
        recent_median_rating = float(np.median(recent_ratings))
    else:
        median_rating = np.nan
        recent_median_rating = np.nan

    # Count topic practice from the user's accepted problems
    topic_counts = Counter()

    for row in rows:
        problem_id = row.problem

        if problem_id not in catalogue:
            continue

        for tag in catalogue[problem_id]["tags"]:
            topic_counts[tag] += 1

    profile = {
        "historical_solved_count": len(rows),
        "historical_median_rating": median_rating,
        "recent_median_rating": recent_median_rating,
        "recent_rating_change": (
            recent_median_rating - median_rating
            if pd.notna(recent_median_rating)
            and pd.notna(median_rating)
            else 0.0
        ),
        "distinct_topics": len(topic_counts),
    }

    for topic in all_topics:
        profile[f"topic_count_{topic}"] = topic_counts[topic]

    return profile


user_profiles = {
    user: build_user_profile(history)
    for user, history in history_train_map.items()
}

user_profiles_df = (
    pd.DataFrame.from_dict(user_profiles, orient="index")
    .rename_axis("user")
    .reset_index()
)

print("User profiles created:", len(user_profiles_df))
print("Profile feature count:", len(user_profiles_df.columns) - 1)

display(user_profiles_df.head())

User profiles created: 300
Profile feature count: 43


,user,historical_solved_count,historical_median_rating,recent_median_rating,recent_rating_change,distinct_topics,topic_count_*special,topic_count_2-sat,topic_count_binary search,topic_count_bitmasks,...,topic_count_number theory,topic_count_probabilities,topic_count_schedules,topic_count_shortest paths,topic_count_sortings,topic_count_string suffix structures,topic_count_strings,topic_count_ternary search,topic_count_trees,topic_count_two pointers
0,user10000,108,800.0,800.0,0.0,20,2,0,2,4,...,3,1,0,0,8,0,17,0,0,2
1,user10046,24,800.0,800.0,0.0,9,1,0,0,0,...,0,0,0,0,2,0,4,0,0,0
2,user10087,81,800.0,850.0,50.0,17,2,0,0,1,...,5,0,0,1,5,0,17,0,0,0
3,user101,381,1100.0,1000.0,-100.0,29,2,0,23,13,...,35,1,0,3,61,0,46,2,3,21
4,user10149,36,1700.0,1900.0,200.0,22,0,0,4,2,...,1,0,0,1,6,2,3,0,1,4


In [105]:

# Join user profiles and add candidate-specific topic preference

# Test profiles must use the test-time history, not training-time history.
test_user_profiles = {
    user: build_user_profile(history)
    for user, history in history_test_map.items()
}

test_user_profiles_df = (
    pd.DataFrame.from_dict(test_user_profiles, orient="index")
    .rename_axis("user")
    .reset_index()
)

# Avoid duplicating historical_solved_count, which already exists
# in the candidate datasets.
profile_columns = [
    col for col in user_profiles_df.columns
    if col not in ["user", "historical_solved_count"]
]

train_profile_lookup = user_profiles_df.set_index("user")
test_profile_lookup = test_user_profiles_df.set_index("user")

train_personalized_df = rebuilt_train_df.merge(
    train_profile_lookup[profile_columns],
    left_on="user",
    right_index=True,
    how="left",
    validate="many_to_one"
)

test_personalized_df = rebuilt_test_df.merge(
    test_profile_lookup[profile_columns],
    left_on="user",
    right_index=True,
    how="left",
    validate="many_to_one"
)

# Calculate how strongly each candidate matches this user's
# historical topic preferences.
topic_count_columns = [f"topic_count_{topic}" for topic in all_topics]

def personalized_topic_score(user, problem_id, profile_lookup):
    if user not in profile_lookup.index:
        return 0.0

    tags = catalogue.get(problem_id, {}).get("tags", [])
    if not tags:
        return 0.0

    profile = profile_lookup.loc[user]
    total = sum(float(profile[col]) for col in topic_count_columns)

    if total == 0:
        return 0.0

    return np.mean([
        float(profile[f"topic_count_{tag}"]) / total
        if f"topic_count_{tag}" in profile.index
        else 0.0
        for tag in tags
    ])

train_personalized_df["personalized_topic_score"] = [
    personalized_topic_score(user, problem, train_profile_lookup)
    for user, problem in zip(
        train_personalized_df["user"],
        train_personalized_df["problem"]
    )
]

test_personalized_df["personalized_topic_score"] = [
    personalized_topic_score(user, problem, test_profile_lookup)
    for user, problem in zip(
        test_personalized_df["user"],
        test_personalized_df["problem"]
    )
]

print("Training shape:", train_personalized_df.shape)
print("Testing shape:", test_personalized_df.shape)
print(
    "Missing training profile values:",
    int(train_personalized_df[profile_columns].isna().sum().sum())
)
print(
    "Missing testing profile values:",
    int(test_personalized_df[profile_columns].isna().sum().sum())
)

display(
    train_personalized_df[
        ["user", "problem", "label",
         "historical_median_rating", "recent_median_rating",
         "personalized_topic_score"]
    ].head()
)

Training shape: (123018, 53)
Testing shape: (140377, 53)
Missing training profile values: 0
Missing testing profile values: 0


,user,problem,label,historical_median_rating,recent_median_rating,personalized_topic_score
0,user12423,1817B,1,2000.0,2000.0,0.050750
1,user12423,1810B,1,2000.0,2000.0,0.064510
2,user12423,1835D,1,2000.0,2000.0,0.052476
3,user12423,1841E,1,2000.0,2000.0,0.088380
4,user12423,1790C,1,2000.0,2000.0,0.078188


In [106]:

# Train and evaluate the personalized LightGBM ranker

import lightgbm as lgb

# Candidate-specific difficulty gap relative to recent skill
for df in [train_personalized_df, test_personalized_df]:
    df["recent_rating_gap"] = (
        df["problem_rating"] - df["recent_median_rating"]
    )
    df["absolute_recent_rating_gap"] = df["recent_rating_gap"].abs()

# Original features + new personalized features
personalized_feature_columns = feature_columns + [
    "historical_median_rating",
    "recent_median_rating",
    "recent_rating_change",
    "distinct_topics",
    "personalized_topic_score",
    "recent_rating_gap",
    "absolute_recent_rating_gap",
] + topic_count_columns

# Sort by user so ranking groups align with their rows
train_model_df = (
    train_personalized_df
    .sort_values("user", kind="stable")
    .reset_index(drop=True)
)

test_model_df = (
    test_personalized_df
    .sort_values("user", kind="stable")
    .reset_index(drop=True)
)

X_train_personalized = train_model_df[personalized_feature_columns]
y_train_personalized = train_model_df["label"]

X_test_personalized = test_model_df[personalized_feature_columns]
y_test_personalized = test_model_df["label"]

train_groups_personalized = (
    train_model_df.groupby("user", sort=False).size().tolist()
)

personalized_ranker = lgb.LGBMRanker(
    objective="lambdarank",
    metric="ndcg",
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    verbosity=-1
)

personalized_ranker.fit(
    X_train_personalized,
    y_train_personalized,
    group=train_groups_personalized
)

personalized_scores = personalized_ranker.predict(
    X_test_personalized
)

personalized_results = ranking_metrics(
    test_model_df,
    personalized_scores,
    k=10
)

print("Personalized model results:")
print(personalized_results)

print("\nFeature count:", len(personalized_feature_columns))
print("Training groups:", len(train_groups_personalized))
print("Test users:", test_model_df["user"].nunique())

display(
    pd.DataFrame({
        "feature": personalized_feature_columns,
        "importance": personalized_ranker.feature_importances_
    }).sort_values("importance", ascending=False).head(15)
)

Personalized model results:
{'NDCG@10': np.float64(0.4863240551286374), 'Recall@10': np.float64(0.1120135759739556)}

Feature count: 52
Training groups: 270
Test users: 298


,feature,importance
11,personalized_topic_score,1734
0,problem_rating,898
4,problem_tag_count,473
1,rating_gap,465
12,recent_rating_gap,384
5,familiar_tag_overlap,325
13,absolute_recent_rating_gap,263
37,topic_count_implementation,261
9,recent_rating_change,260
8,recent_median_rating,211


In [107]:
# Diagnose personalized ranking vs rating-only ranking
# using the exact same test candidates

rating_only_scores = -test_model_df["rating_gap"].abs().to_numpy()

rating_only_results = ranking_metrics(
    test_model_df, rating_only_scores, k=10
)

personalized_results_check = ranking_metrics(
    test_model_df, personalized_scores, k=10
)

diagnostics = pd.DataFrame([
    {
        "model": "Rating-only",
        "NDCG@10": rating_only_results["NDCG@10"],
        "Recall@10": rating_only_results["Recall@10"],
    },
    {
        "model": "Personalized LightGBM",
        "NDCG@10": personalized_results_check["NDCG@10"],
        "Recall@10": personalized_results_check["Recall@10"],
    },
])

display(diagnostics)

# Check score separation between positives and negatives
score_check = test_model_df[["label"]].copy()
score_check["personalized_score"] = personalized_scores
score_check["rating_only_score"] = rating_only_scores

display(
    score_check.groupby("label")[
        ["personalized_score", "rating_only_score"]
    ].agg(["mean", "median"])
)

# Check what each model puts in its top 10 for each user
def top_k_positive_rate(df, scores, k=10):
    ranked = df[["user", "label"]].copy()
    ranked["score"] = scores
    ranked = ranked.sort_values(
        ["user", "score"], ascending=[True, False]
    )
    top = ranked.groupby("user", sort=False).head(k)
    return float(top["label"].mean())

print(
    "Rating-only top-10 positive rate:",
    round(top_k_positive_rate(test_model_df, rating_only_scores), 4)
)
print(
    "Personalized top-10 positive rate:",
    round(top_k_positive_rate(test_model_df, personalized_scores), 4)
)

,model,NDCG@10,Recall@10
0,Rating-only,0.701341,0.125359
1,Personalized LightGBM,0.486324,0.112014


personalized_score           rating_only_score       
                    mean    median              mean median
label                                                      
0              -2.269327 -1.877604       -905.947350 -800.0
1              -0.780153 -0.768626       -397.044335 -300.0

Rating-only top-10 positive rate: 0.6238
Personalized top-10 positive rate: 0.4785


In [108]:
# Validation experiment: blend rating similarity with ML ranking
# The held-out users come from training data, not the test set.

import random
import numpy as np
import pandas as pd
import lightgbm as lgb

rng = random.Random(42)
users = sorted(train_personalized_df["user"].unique())
rng.shuffle(users)

split_at = int(len(users) * 0.8)
fit_users = set(users[:split_at])
valid_users = set(users[split_at:])

fit_df = (
    train_personalized_df[
        train_personalized_df["user"].isin(fit_users)
    ]
    .sort_values("user", kind="stable")
    .reset_index(drop=True)
)

valid_df = (
    train_personalized_df[
        train_personalized_df["user"].isin(valid_users)
    ]
    .sort_values("user", kind="stable")
    .reset_index(drop=True)
)

validation_ranker = lgb.LGBMRanker(
    objective="lambdarank",
    metric="ndcg",
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    verbosity=-1
)

validation_ranker.fit(
    fit_df[personalized_feature_columns],
    fit_df["label"],
    group=fit_df.groupby("user", sort=False).size().tolist()
)

ml_valid_scores = validation_ranker.predict(
    valid_df[personalized_feature_columns]
)

rating_valid_scores = -valid_df["rating_gap"].abs().to_numpy()

# Convert both signals to within-user percentile ranks so their
# different numerical scales do not distort the blend.
rank_df = valid_df[["user", "label"]].copy()
rank_df["ml_score"] = ml_valid_scores
rank_df["rating_score"] = rating_valid_scores

for col in ["ml_score", "rating_score"]:
    rank_df[col + "_rank"] = rank_df.groupby("user")[col].rank(
        method="average", pct=True
    )

results = []

for ml_weight in [0.0, 0.25, 0.5, 0.75, 1.0]:
    combined_scores = (
        ml_weight * rank_df["ml_score_rank"].to_numpy()
        + (1 - ml_weight) * rank_df["rating_score_rank"].to_numpy()
    )

    metrics = ranking_metrics(valid_df, combined_scores, k=10)

    results.append({
        "ML_weight": ml_weight,
        "Rating_weight": 1 - ml_weight,
        "NDCG@10": float(metrics["NDCG@10"]),
        "Recall@10": float(metrics["Recall@10"]),
    })

validation_results = pd.DataFrame(results).sort_values(
    "NDCG@10", ascending=False
)

display(validation_results)
print("Fit users:", len(fit_users))
print("Validation users:", len(valid_users))

,ML_weight,Rating_weight,NDCG@10,Recall@10
0,0.00,1.00,0.711751,0.120147
3,0.75,0.25,0.572346,0.111750
4,1.00,0.00,0.570627,0.119722
2,0.50,0.50,0.521519,0.099695
1,0.25,0.75,0.445408,0.086635


Fit users: 216
Validation users: 54


In [109]:
# Step 1: Check the current project state before rebuilding

from pathlib import Path

model_dir = Path("models")
if not model_dir.exists():
    model_dir = Path("backend/ml/models")

print("Training data:", rebuilt_train_df.shape)
print("Testing data:", rebuilt_test_df.shape)
print("Users in training:", rebuilt_train_df["user"].nunique())
print("Users in testing:", rebuilt_test_df["user"].nunique())
print("Problems in catalogue:", len(catalogue))
print("Available topics:", len(all_topics))

print("\nExisting model artifacts:")
for path in [
    model_dir / "dsaforge_ranker.txt",
    model_dir / "feature_columns.json",
]:
    print(f"{path}: {'EXISTS' if path.exists() else 'NOT FOUND'}")

print("\nCurrent ranking baseline:")
print("Rating-only NDCG@10: 0.701341")
print("Rating-only Recall@10: 0.125359")

Training data: (123018, 10)
Testing data: (140377, 10)
Users in training: 270
Users in testing: 298
Problems in catalogue: 10990
Available topics: 38

Existing model artifacts:
backend\ml\models\dsaforge_ranker.txt: EXISTS
backend\ml\models\feature_columns.json: EXISTS

Current ranking baseline:
Rating-only NDCG@10: 0.701341
Rating-only Recall@10: 0.125359


In [110]:
# Clean personalized ranking model: candidate-specific features only

import lightgbm as lgb
import pandas as pd
import numpy as np

# Exclude user-constant profile columns to reduce noise.
final_features = [
    "problem_rating",
    "rating_gap",
    "absolute_rating_gap",
    "problem_tag_count",
    "familiar_tag_overlap",
    "familiar_tag_ratio",
    "personalized_topic_score",
    "recent_rating_gap",
    "absolute_recent_rating_gap",
]

train_final = (
    train_personalized_df
    .sort_values("user", kind="stable")
    .reset_index(drop=True)
)

test_final = (
    test_personalized_df
    .sort_values("user", kind="stable")
    .reset_index(drop=True)
)

X_train_final = train_final[final_features]
y_train_final = train_final["label"]

X_test_final = test_final[final_features]

train_groups_final = (
    train_final.groupby("user", sort=False).size().tolist()
)

final_ranker = lgb.LGBMRanker(
    objective="lambdarank",
    metric="ndcg",
    n_estimators=300,
    learning_rate=0.03,
    num_leaves=15,
    min_child_samples=50,
    reg_lambda=2.0,
    random_state=42,
    verbosity=-1,
)

final_ranker.fit(
    X_train_final,
    y_train_final,
    group=train_groups_final,
)

final_scores = final_ranker.predict(X_test_final)

ml_metrics = ranking_metrics(test_final, final_scores, k=10)

rating_scores_final = -test_final["rating_gap"].abs().to_numpy()
rating_metrics_final = ranking_metrics(
    test_final, rating_scores_final, k=10
)

print("Clean personalized ranker:")
print({k: round(float(v), 5) for k, v in ml_metrics.items()})

print("\nRating-only comparison:")
print({k: round(float(v), 5) for k, v in rating_metrics_final.items()})

print("\nFeature importance:")
display(
    pd.DataFrame({
        "feature": final_features,
        "importance": final_ranker.feature_importances_,
    }).sort_values("importance", ascending=False)
)

Clean personalized ranker:
{'NDCG@10': 0.49316, 'Recall@10': 0.10646}

Rating-only comparison:
{'NDCG@10': 0.70134, 'Recall@10': 0.12536}

Feature importance:


,feature,importance
6,personalized_topic_score,1179
0,problem_rating,913
7,recent_rating_gap,446
1,rating_gap,376
3,problem_tag_count,340
4,familiar_tag_overlap,323
2,absolute_rating_gap,239
8,absolute_recent_rating_gap,229
5,familiar_tag_ratio,155


In [111]:

# Alternative: LightGBM classifier for personalized candidate scoring

from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score

personalized_classifier = LGBMClassifier(
    objective="binary",
    n_estimators=300,
    learning_rate=0.03,
    num_leaves=15,
    min_child_samples=50,
    reg_lambda=2.0,
    class_weight="balanced",
    random_state=42,
    verbosity=-1,
    n_jobs=-1,
)

personalized_classifier.fit(
    X_train_final,
    y_train_final,
)

# Positive-class probability becomes the candidate ranking score.
classifier_scores = personalized_classifier.predict_proba(
    X_test_final
)[:, 1]

classifier_metrics = ranking_metrics(
    test_final,
    classifier_scores,
    k=10,
)

rating_metrics = ranking_metrics(
    test_final,
    -test_final["rating_gap"].abs().to_numpy(),
    k=10,
)

print("Personalized classifier:")
print({
    key: round(float(value), 5)
    for key, value in classifier_metrics.items()
})

print("\nRating-only baseline:")
print({
    key: round(float(value), 5)
    for key, value in rating_metrics.items()
})

print(
    "\nTest ROC-AUC:",
    round(roc_auc_score(test_final["label"], classifier_scores), 5)
)

Personalized classifier:
{'NDCG@10': 0.53778, 'Recall@10': 0.10964}

Rating-only baseline:
{'NDCG@10': 0.70134, 'Recall@10': 0.12536}

Test ROC-AUC: 0.77703


In [112]:

# Recommendation function using the current personalized ranker

def recommend_personalized(
    user_id,
    history_map,
    tag_map,
    catalogue,
    top_n=10,
):
    history = history_map.get(user_id, [])

    if not history:
        return pd.DataFrame(
            columns=["problem", "rating", "tags", "score"]
        )

    past_ratings = [
        float(row.problem_rating)
        for row in history
        if pd.notna(row.problem_rating)
    ]

    if not past_ratings:
        return pd.DataFrame(
            columns=["problem", "rating", "tags", "score"]
        )

    median_rating = float(np.median(past_ratings))
    recent_ratings = past_ratings[-20:]
    recent_median = float(np.median(recent_ratings))

    solved = {row.problem for row in history}
    user_tags = tag_map.get(user_id, {})
    total_tag_count = sum(user_tags.values())

    candidates = []

    for problem_id, details in catalogue.items():
        if problem_id in solved:
            continue

        rating = details.get("rating")
        tags = details.get("tags", [])

        if rating is None or pd.isna(rating) or not tags:
            continue

        overlap = sum(
            1 for tag in tags
            if user_tags.get(tag, 0) > 0
        )

        topic_score = (
            np.mean([
                user_tags.get(tag, 0) / total_tag_count
                for tag in tags
            ])
            if total_tag_count > 0 else 0.0
        )

        candidates.append({
            "user": user_id,
            "problem": problem_id,
            "problem_rating": float(rating),
            "rating_gap": float(rating) - median_rating,
            "absolute_rating_gap": abs(float(rating) - median_rating),
            "problem_tag_count": len(tags),
            "familiar_tag_overlap": overlap,
            "familiar_tag_ratio": overlap / len(tags),
            "personalized_topic_score": topic_score,
            "recent_rating_gap": float(rating) - recent_median,
            "absolute_recent_rating_gap": abs(float(rating) - recent_median),
            "rating": float(rating),
            "tags": ", ".join(tags),
        })

    if not candidates:
        return pd.DataFrame(
            columns=["problem", "rating", "tags", "score"]
        )

    candidate_df = pd.DataFrame(candidates)

    candidate_df["score"] = final_ranker.predict(
        candidate_df[final_features]
    )

    return (
        candidate_df
        .sort_values(
            ["score", "problem"],
            ascending=[False, True]
        )
        .drop_duplicates("problem")
        [["problem", "rating", "tags", "score"]]
        .head(top_n)
        .reset_index(drop=True)
    )


# Test on one user with known history
sample_user = next(iter(history_test_map))

sample_recommendations = recommend_personalized(
    user_id=sample_user,
    history_map=history_test_map,
    tag_map=tag_test_map,
    catalogue=catalogue,
    top_n=10,
)

print("User:", sample_user)
print("Recommendations returned:", len(sample_recommendations))
display(sample_recommendations)

User: user10000
Recommendations returned: 10


,problem,rating,tags,score
0,2247A,800.0,"constructive algorithms, dp, number theory",2.133993
1,1807G1,1100.0,"brute force, data structures, dp, greedy, impl...",1.386462
2,1688A,800.0,"bitmasks, brute force",1.240250
3,1698A,800.0,"bitmasks, brute force",1.240250
4,1805A,800.0,"bitmasks, brute force",1.240250
5,1844A,800.0,"constructive algorithms, games",1.240250
6,894A,800.0,"brute force, dp",1.240250
7,1454A,800.0,"constructive algorithms, probabilities",1.227281
8,2143A,800.0,"brute force, two pointers",1.217431
9,1462B,800.0,"dp, implementation, strings",1.071765


In [113]:

# Enrich recommendations with problem names and Codeforces URLs

recommendations_with_details = sample_recommendations.merge(
    problem_details[
        ["problem_id", "problem_name", "problem_rating"]
    ].drop_duplicates("problem_id"),
    left_on="problem",
    right_on="problem_id",
    how="left",
    suffixes=("", "_catalogue"),
    validate="many_to_one",
)

# Standard Codeforces problem IDs look like 2247A or 1807G1.
parsed_ids = recommendations_with_details["problem"].str.extract(
    r"^(\d+)([A-Z]\d*)$"
)

recommendations_with_details["problem_url"] = (
    "https://codeforces.com/problemset/problem/"
    + parsed_ids[0].fillna("")
    + "/"
    + parsed_ids[1].fillna("")
)

recommendations_with_details = recommendations_with_details.rename(
    columns={"problem_name": "name"}
)

# Validate recommendation quality and output
history_solved = {
    row.problem
    for row in history_test_map.get(sample_user, [])
}

print("User:", sample_user)
print("Recommendation count:", len(recommendations_with_details))
print(
    "Already-solved problems:",
    recommendations_with_details["problem"].isin(history_solved).sum()
)
print(
    "Duplicate problems:",
    recommendations_with_details["problem"].duplicated().sum()
)
print(
    "Missing problem names:",
    recommendations_with_details["name"].isna().sum()
)
print(
    "Missing URLs:",
    recommendations_with_details["problem_url"].eq(
        "https://codeforces.com/problemset/problem//"
    ).sum()
)

display(
    recommendations_with_details[
        ["problem", "name", "rating", "tags", "problem_url", "score"]
    ]
)

User: user10000
Recommendation count: 10
Already-solved problems: 0
Duplicate problems: 0
Missing problem names: 0
Missing URLs: 0


,problem,name,rating,tags,problem_url,score
0,2247A,Zero Sum,800.0,"constructive algorithms, dp, number theory",https://codeforces.com/problemset/problem/2247/A,2.133993
1,1807G1,Subsequence Addition (Easy Version),1100.0,"brute force, data structures, dp, greedy, impl...",https://codeforces.com/problemset/problem/1807/G1,1.386462
2,1688A,Cirno's Perfect Bitmasks Classroom,800.0,"bitmasks, brute force",https://codeforces.com/problemset/problem/1688/A,1.240250
3,1698A,XOR Mixup,800.0,"bitmasks, brute force",https://codeforces.com/problemset/problem/1698/A,1.240250
4,1805A,We Need the Zero,800.0,"bitmasks, brute force",https://codeforces.com/problemset/problem/1805/A,1.240250
5,1844A,Subtraction Game,800.0,"constructive algorithms, games",https://codeforces.com/problemset/problem/1844/A,1.240250
6,894A,QAQ,800.0,"brute force, dp",https://codeforces.com/problemset/problem/894/A,1.240250
7,1454A,Special Permutation,800.0,"constructive algorithms, probabilities",https://codeforces.com/problemset/problem/1454/A,1.227281
8,2143A,All Lengths Subtraction,800.0,"brute force, two pointers",https://codeforces.com/problemset/problem/2143/A,1.217431
9,1462B,Last Year's Substring,800.0,"dp, implementation, strings",https://codeforces.com/problemset/problem/1462/B,1.071765


In [114]:

# Save the current personalized model as a separate artifact

import json
from pathlib import Path

model_dir = Path("backend/ml/models")
model_dir.mkdir(parents=True, exist_ok=True)

model_path = model_dir / "personalized_ranker_v1.txt"
features_path = model_dir / "personalized_feature_columns.json"

# Save model in LightGBM's native format
final_ranker.booster_.save_model(str(model_path))

# Save feature order required at inference time
with open(features_path, "w", encoding="utf-8") as f:
    json.dump(final_features, f, indent=2)

print("Model saved:", model_path.resolve())
print("Feature list saved:", features_path.resolve())
print("Features:", len(final_features))
print("Model size (KB):", round(model_path.stat().st_size / 1024, 2))
print("Feature file exists:", features_path.exists())
print("Original model preserved:", (model_dir / "dsaforge_ranker.txt").exists())

Model saved: C:\Users\GARIMA YADUVANSHI\OneDrive\Desktop\DSAForge\DSAForge\backend\ml\models\personalized_ranker_v1.txt
Feature list saved: C:\Users\GARIMA YADUVANSHI\OneDrive\Desktop\DSAForge\DSAForge\backend\ml\models\personalized_feature_columns.json
Features: 9
Model size (KB): 526.49
Feature file exists: True
Original model preserved: True


# Section: Curriculum-Adaptive ML Recommender

### 1. Problem Diagnosis with the Previous Baseline Model:
- The previous model in earlier sections used features like `familiar_tag_overlap` and `personalized_topic_score`.
- This caused a severe **repetition trap**: if a student solved many Array or Implementation problems, the model kept recommending *more* Arrays and Implementation problems, even when the student had already mastered Easy, Medium, and Hard tiers!
- It failed to introduce the student to the next topics in their DSA learning progression.

### 2. The Solution: Curriculum-Adaptive ML Ranker
1. **Topic Mastery & Difficulty Breadth**: Tracks solved count and difficulty tiers (Easy, Medium, Hard) per topic to identify mastered topics.
2. **Frontier & Weakness Detection**: Identifies struggling topics (high wrong attempts) and next frontier topics following standard DSA progression.
3. **Curriculum-Adaptive LightGBM Ranker**: Trained to optimize learning growth, penalizing already saturated/mastered topics and prioritizing next-step topics within the student's reach.


In [ ]:
from backend.ml.user_profiler import UserProfile
from backend.ml.recommender_engine import RecommenderEngine

# Demonstrate the exact scenario: Student has mastered Arrays across Easy, Medium, and Hard
profile = UserProfile('student_array_master')
activities = [
    {'title': 'Contains Duplicate', 'topic': 'Arrays', 'difficulty': 'Easy', 'pattern': 'Hash Set', 'rating': 900},
    {'title': 'Two Sum', 'topic': 'Arrays', 'difficulty': 'Easy', 'pattern': 'Hash Map', 'rating': 1000},
    {'title': '3Sum', 'topic': 'Arrays', 'difficulty': 'Medium', 'pattern': 'Two Pointers', 'rating': 1400},
    {'title': 'Subarray Sum Equals K', 'topic': 'Arrays', 'difficulty': 'Medium', 'pattern': 'Prefix Sum', 'rating': 1450},
    {'title': 'First Missing Positive', 'topic': 'Arrays', 'difficulty': 'Hard', 'pattern': 'Array Indexing', 'rating': 1800},
    {'title': 'Trapping Rain Water', 'topic': 'Arrays', 'difficulty': 'Hard', 'pattern': 'Two Pointers', 'rating': 1850},
]
profile.analyze_activities(activities)

print('=== USER PROFILE ===')
print('Mastered Topics:', profile.get_mastered_topics())
print('Next Frontier:', [f['topic'] for f in profile.get_next_frontier_topics()[:5]])

engine = RecommenderEngine()
res = engine.recommend_for_profile(profile, top_n=6)

print('\n=== TOP NEXT PROBLEM RECOMMENDATIONS ===')
for i, r in enumerate(res['recommendations'], 1):
    print(f"{i}. [{r['difficulty']} - {r['rating']}] {r['title']} ({r['topic']})")
    print(f"   Reason: {r['reason']}")


In [ ]:
# Benchmark Comparison: Old Model vs. New Curriculum-Adaptive Model
from backend.ml.evaluate_curriculum_vs_baseline import run_comparison
run_comparison(test_users=10)
